**ONLINE** **SETTING**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
import statsmodels.api as sm

In [ ]:
from google.colab import files
uploaded = files.upload()
csv_path = "BPIC2012_final_online_daily_time_series.csv"
df = pd.read_csv(csv_path)

In [ ]:
df.head()

In [ ]:
# Sort by time if not already sorted
df = df.sort_index()

# 🧪 Hold out the last 20% as final test set
n_test = int(len(df) * 0.20)
df_test_final = df.tail(n_test).copy()
df_train_full = df.drop(df_test_final.index).copy()

df_test_final

In [ ]:
df_train_full

In [ ]:
#feature_engineering.py
import pandas as pd
import numpy as np
from scipy.signal import find_peaks

def feature_engineering(
    df,
    target_var,
    horizon=1,
    target_lags=range(1, 8),
    actor_lags=range(1, 8),
    rolling_windows=[3, 7, 14],
    smooth_window=3
):
    df_model = df.copy()

    df_model = df.copy()

    # Remove trailing days where no cases are running
    core_kpis = ["WIP", "Avg_Elapsed_Time", "Avg_Remaining_Time"]
    existing_kpis = [k for k in core_kpis if k in df_model.columns]

    while len(df_model) > 1 and df_model[existing_kpis].iloc[-1].sum() == 0:
        df_model = df_model.iloc[:-1]

    # -----------------------------
    # 1. Define smoothed change target
    # -----------------------------
    # Predict change from today to t + horizon
    df_model["target"] = df_model[target_var].shift(-horizon) - df_model[target_var]

    # -----------------------------
    # 2. Decide which KPI series can be used as baseline input
    # -----------------------------
    if target_var == "Avg_Elapsed_Time":
        baseline_kpi_vars = ["Avg_Elapsed_Time"]

    elif target_var == "WIP":
        baseline_kpi_vars = ["WIP"]

    elif target_var == "Avg_Remaining_Time":
        baseline_kpi_vars = ["Avg_Elapsed_Time"]

    else:
        raise ValueError(f"Unsupported target_var: {target_var}")

    baseline_features = []

    # -----------------------------
    # 3. Baseline KPI features
    # -----------------------------
    for var in baseline_kpi_vars:
        if var not in df_model.columns:
            continue

        for lag in target_lags:
            col = f"{var}_lag{lag}"
            df_model[col] = df_model[var].shift(lag)
            baseline_features.append(col)

        for window in rolling_windows:
            mean_col = f"{var}_rolling_mean{window}"
            std_col = f"{var}_rolling_std{window}"
            max_col = f"{var}_rolling_max{window}"

            df_model[mean_col] = df_model[var].rolling(window).mean().shift(1)
            df_model[std_col] = df_model[var].rolling(window).std().shift(1)
            df_model[max_col] = df_model[var].rolling(window).max().shift(1)

            baseline_features += [mean_col, std_col, max_col]

        zcol = f"{var}_zscore7"
        df_model[zcol] = (
            (df_model[var].shift(1) - df_model[f"{var}_rolling_mean7"]) /
            df_model[f"{var}_rolling_std7"]
        )
        baseline_features.append(zcol)

        peaks, _ = find_peaks(df_model[var].fillna(0), distance=7)
        peak_col = f"{var}_peak_flag"
        df_model[peak_col] = 0
        if len(peaks) > 0:
            df_model.iloc[peaks, df_model.columns.get_loc(peak_col)] = 1
        baseline_features.append(peak_col)

    # -----------------------------
    # 4. Actor features
    # -----------------------------
    actor_vars = [
        "Cum_Count_C", "Cum_Count_I", "Cum_Count_HI", "Cum_Count_HB",
        "Cum_Time_C_seconds", "Cum_Time_I_seconds",
        "Cum_Time_HI_seconds", "Cum_Time_HB_seconds"
    ]

    actor_features = []

    for var in actor_vars:
        if var in df_model.columns:
            actor_features.append(var)

            for lag in actor_lags:
                col = f"{var}_lag{lag}"
                df_model[col] = df_model[var].shift(lag)
                actor_features.append(col)

            for window in rolling_windows:
                mean_col = f"{var}_rolling_mean{window}"
                std_col = f"{var}_rolling_std{window}"
                max_col = f"{var}_rolling_max{window}"

                df_model[mean_col] = df_model[var].rolling(window).mean().shift(1)
                df_model[std_col] = df_model[var].rolling(window).std().shift(1)
                df_model[max_col] = df_model[var].rolling(window).max().shift(1)

                actor_features += [mean_col, std_col, max_col]

            zcol = f"{var}_zscore7"
            df_model[zcol] = (
                (df_model[var].shift(1) - df_model[f"{var}_rolling_mean7"]) /
                df_model[f"{var}_rolling_std7"]
            )
            actor_features.append(zcol)

    # -----------------------------
    # 5. Cleanup
    # -----------------------------
    df_model.dropna(inplace=True)

    baseline_features = list(dict.fromkeys(baseline_features))
    actor_features = [f for f in dict.fromkeys(actor_features) if f not in baseline_features]

    return df_model.copy(), baseline_features, actor_features



In [ ]:
# ============================================================
# CHECK WEEKLY SEASONALITY AND COMPARE ARIMAX VS SARIMAX
# Uses TRAINING DATA ONLY for the model-choice decision
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error

# ------------------------------------------------------------
# 1. Upload and load daily time-series CSV
# ------------------------------------------------------------

uploaded = files.upload()
csv_path = list(uploaded.keys())[0]

df = pd.read_csv(csv_path)

print("Columns in uploaded file:")
print(df.columns.tolist())

# Automatically detect a likely date column.
possible_date_columns = [
    col for col in df.columns
    if col.lower() in ["date", "day", "timestamp", "time", "datetime"]
    or "date" in col.lower()
    or "day" in col.lower()
]

if len(possible_date_columns) > 0:
    DATE_COLUMN = possible_date_columns[0]
    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])
    df = df.sort_values(DATE_COLUMN).set_index(DATE_COLUMN)
    print(f"\nUsing date column: {DATE_COLUMN}")
else:
    # Fall back to the existing row order if no date column is present.
    df = df.copy()
    df.index = pd.RangeIndex(start=0, stop=len(df), step=1)
    print("\nNo date column detected. Using existing row order.")

targets = ["Avg_Elapsed_Time", "Avg_Remaining_Time"]
targets = [target for target in targets if target in df.columns]

if len(targets) == 0:
    raise ValueError("No Avg_Elapsed_Time or Avg_Remaining_Time columns found.")

# Remove trailing empty days if present.
while len(df) > 1 and df[targets].iloc[-1].fillna(0).sum() == 0:
    df = df.iloc[:-1]

# ------------------------------------------------------------
# 2. Chronological split: inspect TRAINING data only
# ------------------------------------------------------------

n_test = int(len(df) * 0.20)
df_train = df.iloc[:-n_test].copy()
df_test = df.iloc[-n_test:].copy()

print(f"\nTotal observations: {len(df)}")
print(f"Training observations used for seasonality checks: {len(df_train)}")
print(f"Final hold-out observations not used here: {len(df_test)}")

# ------------------------------------------------------------
# 3. Visual checks: original series, weekday profile, STL, ACF
# ------------------------------------------------------------

if isinstance(df_train.index, pd.DatetimeIndex):
    weekday_names = ["Monday", "Tuesday", "Wednesday", "Thursday",
                     "Friday", "Saturday", "Sunday"]

for target in targets:
    series = df_train[target].dropna()

    print("\n" + "=" * 70)
    print(f"SEASONALITY CHECK: {target}")
    print("=" * 70)

    # Original training series
    plt.figure(figsize=(14, 4))
    plt.plot(series.index, series.values)
    plt.title(f"Training series: {target}")
    plt.xlabel("Day")
    plt.ylabel(target)
    plt.tight_layout()
    plt.show()

    # Mean value by weekday, if a date index exists
    if isinstance(df_train.index, pd.DatetimeIndex):
        weekday_df = pd.DataFrame({target: series})
        weekday_df["weekday"] = weekday_df.index.dayofweek

        weekday_profile = weekday_df.groupby("weekday")[target].mean().reindex(range(7))
        weekday_profile.index = weekday_names

        plt.figure(figsize=(9, 4))
        plt.plot(weekday_profile.index, weekday_profile.values, marker="o")
        plt.title(f"Mean {target} by weekday: training data only")
        plt.xlabel("Weekday")
        plt.ylabel("Mean value")
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()

        print("\nMean by weekday:")
        print(weekday_profile.round(3))

    # ACF with weekly lag markers
    if len(series) > 20:
        max_lags = min(35, len(series) // 2 - 1)

        fig, ax = plt.subplots(figsize=(12, 4))
        plot_acf(series, lags=max_lags, ax=ax)

        for weekly_lag in [7, 14, 21, 28]:
            if weekly_lag <= max_lags:
                ax.axvline(weekly_lag, linestyle="--", alpha=0.5)

        ax.set_title(f"ACF of {target}: dashed lines indicate weekly lags")
        plt.tight_layout()
        plt.show()

        acf_values = [
            series.autocorr(lag=lag)
            for lag in [7, 14, 21, 28]
            if lag < len(series)
        ]
        used_lags = [
            lag for lag in [7, 14, 21, 28]
            if lag < len(series)
        ]

        print("\nAutocorrelation at weekly lags:")
        for lag, value in zip(used_lags, acf_values):
            print(f"Lag {lag}: {value:.3f}")

    # Weekly STL decomposition
    if len(series) >= 28:
        stl_result = STL(series, period=7, robust=True).fit()
        stl_result.plot()
        plt.suptitle(f"Weekly STL decomposition: {target}", y=1.02)
        plt.tight_layout()
        plt.show()

        seasonal_strength = max(
            0,
            1 - np.var(stl_result.resid) /
            np.var(stl_result.seasonal + stl_result.resid)
        )

        print(f"\nWeekly seasonal strength: {seasonal_strength:.3f}")

# ------------------------------------------------------------
# 4. Compare ARIMAX vs weekly SARIMAX by chronological validation
#    This tests the target-change series actually modeled in your code.
# ------------------------------------------------------------

horizons = [1, 3, 7]

# Internal validation split within the training partition only
n_val = max(14, int(len(df_train) * 0.20))
train_inner = df_train.iloc[:-n_val].copy()
val_inner = df_train.iloc[-n_val:].copy()

comparison_rows = []

for target in targets:
    for horizon in horizons:

        # Build the horizon-specific target-change series:
        # Delta_h y(t) = y(t+h) - y(t)
        delta_full = df_train[target].shift(-horizon) - df_train[target]

        delta_train = delta_full.loc[train_inner.index].dropna()
        delta_val = delta_full.loc[val_inner.index].dropna()

        if len(delta_train) < 20 or len(delta_val) < 5:
            print(f"Skipping {target}, h={horizon}: insufficient observations.")
            continue

        # Non-seasonal ARIMAX-type model
        arimax_model = SARIMAX(
            delta_train,
            order=(1, 0, 0),
            seasonal_order=(0, 0, 0, 0),
            trend="c",
            enforce_stationarity=False,
            enforce_invertibility=False
        ).fit(disp=False)

        arimax_pred = arimax_model.forecast(steps=len(delta_val))
        arimax_rmse = np.sqrt(mean_squared_error(delta_val.values, arimax_pred.values))

        # Weekly seasonal SARIMAX candidate
        sarimax_model = SARIMAX(
            delta_train,
            order=(1, 0, 0),
            seasonal_order=(1, 0, 0, 7),
            trend="c",
            enforce_stationarity=False,
            enforce_invertibility=False
        ).fit(disp=False)

        sarimax_pred = sarimax_model.forecast(steps=len(delta_val))
        sarimax_rmse = np.sqrt(mean_squared_error(delta_val.values, sarimax_pred.values))

        better_model = "Weekly SARIMAX" if sarimax_rmse < arimax_rmse else "ARIMAX"

        comparison_rows.append({
            "Target": target,
            "Horizon": horizon,
            "ARIMAX Validation RMSE": arimax_rmse,
            "Weekly SARIMAX Validation RMSE": sarimax_rmse,
            "Preferred Specification": better_model
        })

comparison_df = pd.DataFrame(comparison_rows)

print("\n" + "=" * 70)
print("CHRONOLOGICAL VALIDATION COMPARISON: ARIMAX VS WEEKLY SARIMAX")
print("=" * 70)
print(comparison_df.round(4).to_string(index=False))

# Summary decision
if len(comparison_df) > 0:
    seasonal_wins = (comparison_df["Preferred Specification"] == "Weekly SARIMAX").sum()
    total = len(comparison_df)

    print("\nDecision summary:")
    print(f"Weekly SARIMAX performs better in {seasonal_wins} of {total} target/horizon settings.")

    if seasonal_wins > total / 2:
        print("Weekly seasonal terms appear useful. Consider rerunning the classical-model experiments with SARIMAX.")
    else:
        print("Weekly seasonal terms do not consistently improve validation performance. Keep the non-seasonal model and report it as ARIMAX.")

comparison_df.to_csv("arimax_vs_weekly_sarimax_validation_comparison.csv", index=False)
files.download("arimax_vs_weekly_sarimax_validation_comparison.csv")

# Classical statistical time series forecasting models



In [ ]:
# ============================================================
# CLASSICAL TIME-SERIES MODELS
# 1) SARIMAX
# 2) ARX-HAC Dynamic Regression
#
# Includes:
# - training
# - final holdout testing
# - baseline vs actor-enriched comparison
# - Wilcoxon + Diebold-Mariano style p-tests
# - feature contribution tables with coefficient p-values
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

from scipy.stats import wilcoxon, norm
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_regression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

CLASSICAL_SAVE_DIR = "classical_time_series_results"
os.makedirs(CLASSICAL_SAVE_DIR, exist_ok=True)

horizons = [1, 3, 7]
targets = ["Avg_Elapsed_Time", "Avg_Remaining_Time"]

MAX_FEATURES = 25        # keeps SARIMAX/OLS stable with many engineered actor features
SARIMAX_ORDER = (1, 0, 0)
SARIMAX_SEASONAL_ORDER = (0, 0, 0, 0)  # change to (1,0,0,7) if your daily series has weekly seasonality


# ------------------------------------------------------------
# Utility functions
# ------------------------------------------------------------

def fmt_mean_std(values):
    values = np.asarray(values)
    return f"{np.mean(values):.3f} ± {np.std(values, ddof=1):.3f}"


def regression_feature_screen(X_train, y_train, X_test, max_features=25):
    """
    Cleans, imputes, scales, and selects strongest features using univariate f-statistics.
    Returns DataFrames so statsmodels keeps readable feature names.
    """
    X_train = X_train.replace([np.inf, -np.inf], np.nan).copy()
    X_test = X_test.replace([np.inf, -np.inf], np.nan).copy()

    # Drop all-null and constant columns
    usable_cols = []
    for col in X_train.columns:
        if X_train[col].notna().sum() == 0:
            continue
        if X_train[col].nunique(dropna=True) <= 1:
            continue
        usable_cols.append(col)

    X_train = X_train[usable_cols]
    X_test = X_test[usable_cols]

    if X_train.shape[1] == 0:
        raise ValueError("No usable features after removing null/constant columns.")

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)

    X_train_scaled_np = scaler.fit_transform(X_train_imp)
    X_test_scaled_np = scaler.transform(X_test_imp)

    X_train_scaled = pd.DataFrame(
        X_train_scaled_np,
        columns=X_train.columns,
        index=X_train.index
    )

    X_test_scaled = pd.DataFrame(
        X_test_scaled_np,
        columns=X_train.columns,
        index=X_test.index
    )

    k = min(max_features, X_train_scaled.shape[1])

    try:
        f_vals, _ = f_regression(X_train_scaled, y_train)
        f_vals = np.nan_to_num(f_vals, nan=-np.inf, posinf=-np.inf, neginf=-np.inf)
        selected_idx = np.argsort(f_vals)[::-1][:k]
        selected_features = list(X_train_scaled.columns[selected_idx])
    except Exception:
        selected_features = list(X_train_scaled.columns[:k])

    return (
        X_train_scaled[selected_features],
        X_test_scaled[selected_features],
        selected_features
    )


def evaluate_forecast(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }


def diebold_mariano_hac_pvalue(y_true, pred_baseline, pred_actor, loss="squared", max_lag=1):
    """
    One-sided DM-style test.

    H0: equal predictive accuracy
    H1: baseline has larger loss than actor-enriched model

    Uses a Newey-West / HAC style long-run variance estimate.
    """
    y_true = np.asarray(y_true)
    pred_baseline = np.asarray(pred_baseline)
    pred_actor = np.asarray(pred_actor)

    if loss == "absolute":
        d = np.abs(y_true - pred_baseline) - np.abs(y_true - pred_actor)
    else:
        d = (y_true - pred_baseline) ** 2 - (y_true - pred_actor) ** 2

    d = d[~np.isnan(d)]
    n = len(d)

    if n < 5 or np.std(d) == 0:
        return np.nan, np.nan

    d_centered = d - np.mean(d)

    gamma0 = np.mean(d_centered * d_centered)
    long_run_var = gamma0

    lag_limit = min(max_lag, n - 1)

    for lag in range(1, lag_limit + 1):
        weight = 1 - lag / (lag_limit + 1)
        gamma = np.mean(d_centered[lag:] * d_centered[:-lag])
        long_run_var += 2 * weight * gamma

    if long_run_var <= 0:
        return np.nan, np.nan

    dm_stat = np.mean(d) / np.sqrt(long_run_var / n)

    # one-sided: positive mean loss difference means actor is better
    p_value = 1 - norm.cdf(dm_stat)

    return dm_stat, p_value


def build_feature_contribution_table(
    fitted_result,
    X_test_scaled,
    selected_features,
    model_name,
    target_var,
    horizon,
    feature_set,
    baseline_features,
    actor_features
):
    """
    Contribution approximation for linear classical models:

    contribution_i,t = beta_i * standardized_feature_i,t

    MeanAbsContribution shows the average absolute contribution
    of each feature to the predicted target change.
    """
    params = pd.Series(fitted_result.params)
    pvalues = pd.Series(fitted_result.pvalues)

    rows = []

    for feature in selected_features:
        if feature not in params.index:
            continue

        coef = params[feature]
        pval = pvalues.get(feature, np.nan)

        contrib = X_test_scaled[feature].values * coef

        if feature in actor_features:
            group = "Actor"
        elif feature in baseline_features:
            group = "Baseline"
        else:
            group = "Other"

        rows.append({
            "Model": model_name,
            "Target": target_var,
            "Horizon": horizon,
            "Feature Set": feature_set,
            "Feature": feature,
            "Group": group,
            "Coefficient": coef,
            "p-value": pval,
            "Mean Contribution": np.mean(contrib),
            "Mean Abs Contribution": np.mean(np.abs(contrib)),
            "Direction": "Positive" if coef > 0 else "Negative"
        })

    contribution_df = pd.DataFrame(rows)

    if len(contribution_df) > 0:
        contribution_df = contribution_df.sort_values(
            "Mean Abs Contribution",
            ascending=False
        ).reset_index(drop=True)

    return contribution_df


# ------------------------------------------------------------
# Model fitting functions
# ------------------------------------------------------------

def fit_predict_sarimax(X_train, y_train, X_test):
    """
    SARIMAX with exogenous regressors.
    Predicts the engineered target, i.e. future change.
    """
    model = SARIMAX(
        endog=y_train,
        exog=X_train,
        order=SARIMAX_ORDER,
        seasonal_order=SARIMAX_SEASONAL_ORDER,
        trend="c",
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    fitted = model.fit(disp=False, maxiter=300)

    pred_delta = fitted.forecast(
        steps=len(X_test),
        exog=X_test
    )

    pred_delta = pd.Series(pred_delta).reset_index(drop=True)

    return fitted, pred_delta


def fit_predict_arx_hac(X_train, y_train, X_test, hac_lags=7):
    """
    ARX-HAC Dynamic Regression.

    This is a classical linear forecasting model using the lagged/rolling
    features already created by your feature_engineering function.

    HAC robust covariance gives time-series-aware standard errors and p-values.
    """
    X_train_const = sm.add_constant(X_train, has_constant="add")
    X_test_const = sm.add_constant(X_test, has_constant="add")

    model = sm.OLS(y_train, X_train_const)

    fitted = model.fit(
        cov_type="HAC",
        cov_kwds={"maxlags": hac_lags}
    )

    pred_delta = fitted.predict(X_test_const)
    pred_delta = pd.Series(pred_delta).reset_index(drop=True)

    return fitted, pred_delta


# ------------------------------------------------------------
# Main training + testing loop
# ------------------------------------------------------------

classical_result_rows = []
classical_p_test_rows = []
classical_contribution_tables = []
classical_predictions = {}

for target_var in targets:
    for horizon in horizons:

        print("\n" + "=" * 70)
        print(f"CLASSICAL MODELS | Target: {target_var} | Horizon: {horizon}")
        print("=" * 70)

        # -----------------------------
        # Feature engineering
        # -----------------------------
        df_train_fe, baseline_features, actor_features = feature_engineering(
            df_train_full,
            target_var=target_var,
            horizon=horizon
        )

        df_context = pd.concat([df_train_full.tail(40), df_test_final])

        df_test_fe, _, _ = feature_engineering(
            df_context,
            target_var=target_var,
            horizon=horizon
        )

        df_test_fe = df_test_fe.loc[
            df_test_final.index.intersection(df_test_fe.index)
        ]

        df_train_fe = df_train_fe.dropna().copy()
        df_test_fe = df_test_fe.dropna().copy()

        y_train = df_train_fe["target"].reset_index(drop=True)
        y_test_delta = df_test_fe["target"].reset_index(drop=True)

        current_values = df_test_fe[target_var].reset_index(drop=True)
        y_true = current_values + y_test_delta

        feature_sets = {
            "Baseline": baseline_features,
            "Actor": baseline_features + actor_features
        }

        for feature_set_name, features in feature_sets.items():

            print(f"\n--- Feature set: {feature_set_name} ---")

            X_train_raw = df_train_fe[features].reset_index(drop=True)
            X_test_raw = df_test_fe[features].reset_index(drop=True)

            X_train, X_test, selected_features = regression_feature_screen(
                X_train_raw,
                y_train,
                X_test_raw,
                max_features=MAX_FEATURES
            )

            # -----------------------------
            # Model 1: SARIMAX
            # -----------------------------
            try:
                sarimax_fit, sarimax_pred_delta = fit_predict_sarimax(
                    X_train,
                    y_train,
                    X_test
                )

                sarimax_pred = current_values + sarimax_pred_delta
                sarimax_metrics = evaluate_forecast(y_true, sarimax_pred)

                classical_result_rows.append({
                    "Model": "SARIMAX",
                    "Target": target_var,
                    "Horizon": horizon,
                    "Feature Set": feature_set_name,
                    "N Features": len(selected_features),
                    "RMSE": sarimax_metrics["RMSE"],
                    "MAE": sarimax_metrics["MAE"],
                    "R2": sarimax_metrics["R2"]
                })

                classical_predictions[("SARIMAX", target_var, horizon, feature_set_name)] = {
                    "index": df_test_fe.index,
                    "y_true": y_true,
                    "y_pred": sarimax_pred,
                    "pred_delta": sarimax_pred_delta,
                    "selected_features": selected_features,
                    "fitted_model": sarimax_fit
                }

                sarimax_contrib = build_feature_contribution_table(
                    fitted_result=sarimax_fit,
                    X_test_scaled=X_test,
                    selected_features=selected_features,
                    model_name="SARIMAX",
                    target_var=target_var,
                    horizon=horizon,
                    feature_set=feature_set_name,
                    baseline_features=baseline_features,
                    actor_features=actor_features
                )

                classical_contribution_tables.append(sarimax_contrib)

                print(
                    f"SARIMAX {feature_set_name} | "
                    f"RMSE={sarimax_metrics['RMSE']:.3f}, "
                    f"MAE={sarimax_metrics['MAE']:.3f}, "
                    f"R2={sarimax_metrics['R2']:.3f}"
                )

            except Exception as e:
                print(f"SARIMAX failed for {target_var}, h={horizon}, {feature_set_name}: {e}")

            # -----------------------------
            # Model 2: ARX-HAC
            # -----------------------------
            try:
                arx_fit, arx_pred_delta = fit_predict_arx_hac(
                    X_train,
                    y_train,
                    X_test,
                    hac_lags=max(1, horizon)
                )

                arx_pred = current_values + arx_pred_delta
                arx_metrics = evaluate_forecast(y_true, arx_pred)

                classical_result_rows.append({
                    "Model": "ARX-HAC",
                    "Target": target_var,
                    "Horizon": horizon,
                    "Feature Set": feature_set_name,
                    "N Features": len(selected_features),
                    "RMSE": arx_metrics["RMSE"],
                    "MAE": arx_metrics["MAE"],
                    "R2": arx_metrics["R2"]
                })

                classical_predictions[("ARX-HAC", target_var, horizon, feature_set_name)] = {
                    "index": df_test_fe.index,
                    "y_true": y_true,
                    "y_pred": arx_pred,
                    "pred_delta": arx_pred_delta,
                    "selected_features": selected_features,
                    "fitted_model": arx_fit
                }

                arx_contrib = build_feature_contribution_table(
                    fitted_result=arx_fit,
                    X_test_scaled=X_test,
                    selected_features=selected_features,
                    model_name="ARX-HAC",
                    target_var=target_var,
                    horizon=horizon,
                    feature_set=feature_set_name,
                    baseline_features=baseline_features,
                    actor_features=actor_features
                )

                classical_contribution_tables.append(arx_contrib)

                print(
                    f"ARX-HAC {feature_set_name} | "
                    f"RMSE={arx_metrics['RMSE']:.3f}, "
                    f"MAE={arx_metrics['MAE']:.3f}, "
                    f"R2={arx_metrics['R2']:.3f}"
                )

            except Exception as e:
                print(f"ARX-HAC failed for {target_var}, h={horizon}, {feature_set_name}: {e}")


# ------------------------------------------------------------
# Results dataframe
# ------------------------------------------------------------

classical_results_df = pd.DataFrame(classical_result_rows)

print("\n📊 Classical model holdout results:")
print(classical_results_df.to_markdown(index=False))


# ------------------------------------------------------------
# Baseline vs Actor-Enriched p-tests
# ------------------------------------------------------------

for model_name in ["SARIMAX", "ARX-HAC"]:
    for target_var in targets:
        for horizon in horizons:

            key_b = (model_name, target_var, horizon, "Baseline")
            key_a = (model_name, target_var, horizon, "Actor")

            if key_b not in classical_predictions or key_a not in classical_predictions:
                continue

            y_true = np.asarray(classical_predictions[key_b]["y_true"])
            pred_b = np.asarray(classical_predictions[key_b]["y_pred"])
            pred_a = np.asarray(classical_predictions[key_a]["y_pred"])

            abs_err_b = np.abs(y_true - pred_b)
            abs_err_a = np.abs(y_true - pred_a)

            sq_err_b = (y_true - pred_b) ** 2
            sq_err_a = (y_true - pred_a) ** 2

            # Wilcoxon one-sided:
            # H1: baseline errors > actor-enriched errors
            try:
                w_mae_stat, w_mae_p = wilcoxon(
                    abs_err_b,
                    abs_err_a,
                    alternative="greater"
                )
            except Exception:
                w_mae_stat, w_mae_p = np.nan, np.nan

            try:
                w_rmse_stat, w_rmse_p = wilcoxon(
                    sq_err_b,
                    sq_err_a,
                    alternative="greater"
                )
            except Exception:
                w_rmse_stat, w_rmse_p = np.nan, np.nan

            dm_sq_stat, dm_sq_p = diebold_mariano_hac_pvalue(
                y_true,
                pred_b,
                pred_a,
                loss="squared",
                max_lag=max(1, horizon)
            )

            dm_abs_stat, dm_abs_p = diebold_mariano_hac_pvalue(
                y_true,
                pred_b,
                pred_a,
                loss="absolute",
                max_lag=max(1, horizon)
            )

            rmse_b = np.sqrt(mean_squared_error(y_true, pred_b))
            rmse_a = np.sqrt(mean_squared_error(y_true, pred_a))
            mae_b = mean_absolute_error(y_true, pred_b)
            mae_a = mean_absolute_error(y_true, pred_a)
            r2_b = r2_score(y_true, pred_b)
            r2_a = r2_score(y_true, pred_a)

            classical_p_test_rows.append({
                "Model": model_name,
                "Target": target_var,
                "Horizon": horizon,

                "RMSE Baseline": rmse_b,
                "RMSE Actor": rmse_a,
                "Delta RMSE": rmse_b - rmse_a,

                "MAE Baseline": mae_b,
                "MAE Actor": mae_a,
                "Delta MAE": mae_b - mae_a,

                "R2 Baseline": r2_b,
                "R2 Actor": r2_a,
                "Delta R2": r2_a - r2_b,

                "Wilcoxon MAE p-value": w_mae_p,
                "Wilcoxon Squared Error p-value": w_rmse_p,

                "DM Squared Loss stat": dm_sq_stat,
                "DM Squared Loss p-value": dm_sq_p,

                "DM Absolute Loss stat": dm_abs_stat,
                "DM Absolute Loss p-value": dm_abs_p
            })

classical_p_tests_df = pd.DataFrame(classical_p_test_rows)

print("\n📊 Classical model baseline vs actor-enriched p-tests:")
print(classical_p_tests_df.to_markdown(index=False))


# ------------------------------------------------------------
# Feature contribution tables
# ------------------------------------------------------------

if len(classical_contribution_tables) > 0:
    classical_feature_contributions_df = pd.concat(
        classical_contribution_tables,
        ignore_index=True
    )
else:
    classical_feature_contributions_df = pd.DataFrame()

print("\n🔎 Top classical feature contributions:")
if len(classical_feature_contributions_df) > 0:
    display(
        classical_feature_contributions_df
        .sort_values(["Model", "Target", "Horizon", "Feature Set", "Mean Abs Contribution"],
                     ascending=[True, True, True, True, False])
        .groupby(["Model", "Target", "Horizon", "Feature Set"])
        .head(10)
    )
else:
    print("No contribution table was generated.")


# ------------------------------------------------------------
# Save artifacts
# ------------------------------------------------------------

classical_results_df.to_csv(
    f"{CLASSICAL_SAVE_DIR}/classical_holdout_results.csv",
    index=False
)

classical_p_tests_df.to_csv(
    f"{CLASSICAL_SAVE_DIR}/classical_actor_vs_baseline_p_tests.csv",
    index=False
)

classical_feature_contributions_df.to_csv(
    f"{CLASSICAL_SAVE_DIR}/classical_feature_contributions.csv",
    index=False
)

print(f"\n✅ Saved classical model outputs to: {CLASSICAL_SAVE_DIR}")

# XGBoost

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel, sem, t, wilcoxon

In [ ]:
# Use best config from sweep
best_xgb_params = {
    'n_estimators': 1500,
    'learning_rate': 0.2,
    'max_depth': 3,
    'subsample': 0.9,
    'colsample_bytree': 0.9,
    'gamma': 0,
    'min_child_weight': 1,
    'random_state': 42
}

In [ ]:
horizons=[1,3,7]
targets = ["Avg_Elapsed_Time", "Avg_Remaining_Time"]
all_results = []
all_holdout_results = []
holdout_predictions = {}

In [ ]:
# train_xgboost.py and train_lightgbm.py (shared structure)
# ============================================================
# XGBoost Training with Artifact Saving
# ============================================================

import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import ttest_rel, sem, t, wilcoxon

# -----------------------------
# Output directory
# -----------------------------
SAVE_DIR = "xgb_training_artifacts"
os.makedirs(SAVE_DIR, exist_ok=True)

# -----------------------------
# Best XGBoost configuration
# -----------------------------
# Use best config from sweep
best_xgb_params = {
    'n_estimators': 1500,
    'learning_rate': 0.2,
    'max_depth': 3,
    'subsample': 0.9,
    'colsample_bytree': 0.9,
    'gamma': 0,
    'min_child_weight': 1,
    'random_state': 42
}

horizons = [1, 3, 7]
targets = ["Avg_Elapsed_Time", "Avg_Remaining_Time"]

all_results = []
trained_models = {}
cv_predictions = {}
feature_sets = {}

# -----------------------------
# Helper functions
# -----------------------------
def cohen_d(x, y):
    diff = np.array(x) - np.array(y)
    return diff.mean() / diff.std(ddof=1)

def confidence_interval(data, confidence=0.95):
    m = np.mean(data)
    se_val = sem(data)
    h = se_val * t.ppf((1 + confidence) / 2.0, len(data) - 1)
    return f"{m:.3f} ± {h:.3f}"

def summarize_model_comparison(name, target_name, horizon, rmse_b, rmse_a, mae_b, mae_a, r2_b, r2_a):
    stat, p_rmse = ttest_rel(rmse_b, rmse_a)
    stat_w, p_wilcoxon = wilcoxon(rmse_b, rmse_a)

    return {
        "Model": name,
        "Target": target_name,
        "Horizon": horizon,
        "RMSE Baseline": confidence_interval(rmse_b),
        "RMSE Actor": confidence_interval(rmse_a),
        "RMSE Δ": f"{np.mean(rmse_b) - np.mean(rmse_a):.3f}",
        "MAE Baseline": confidence_interval(mae_b),
        "MAE Actor": confidence_interval(mae_a),
        "MAE Δ": f"{np.mean(mae_b) - np.mean(mae_a):.3f}",
        "R² Baseline": confidence_interval(r2_b),
        "R² Actor": confidence_interval(r2_a),
        "R² Δ": f"{np.mean(r2_a) - np.mean(r2_b):.3f}",
        "p-value RMSE t-test": f"{p_rmse:.4f}",
        "p-value RMSE Wilcoxon": f"{p_wilcoxon:.4f}",
        "Cohen’s d": f"{cohen_d(rmse_b, rmse_a):.3f}"
    }

# -----------------------------
# Cross-validation training
# -----------------------------
for horizon in horizons:
    for target_var in targets:
        print("\n==============================")
        print(f"Running XGBoost | Target: {target_var} | Horizon: {horizon}")
        print("==============================")

        df = df_train_full.copy()
        tscv = TimeSeriesSplit(n_splits=5)

        rmse_b, mae_b, r2_b = [], [], []
        rmse_a, mae_a, r2_a = [], [], []

        all_true, all_pred_b, all_pred_a = [], [], []
        all_err_b, all_err_a = [], []

        imp_b, imp_a = [], []

        for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
            print(f"Fold {fold + 1}")

            df_train_raw = df.iloc[:test_idx[0]].copy()
            df_test_raw = df.iloc[test_idx].copy()

            df_train_fe, baseline_features, actor_features = feature_engineering(
                df_train_raw,
                target_var=target_var,
                horizon=horizon
            )

            df_context = pd.concat([df_train_raw.tail(40), df_test_raw])
            df_test_fe, _, _ = feature_engineering(
                df_context,
                target_var=target_var,
                horizon=horizon
            )
            df_test_fe = df_test_fe.loc[df_test_raw.index.intersection(df_test_fe.index)]

            df_train_fe.dropna(inplace=True)
            df_test_fe.dropna(inplace=True)

            X_train_b = df_train_fe[baseline_features]
            X_test_b = df_test_fe[baseline_features]

            X_train_a = df_train_fe[baseline_features + actor_features]
            X_test_a = df_test_fe[baseline_features + actor_features]

            y_train = df_train_fe["target"]
            y_test = df_test_fe["target"].reset_index(drop=True)

            current_values = df_test_fe[target_var].reset_index(drop=True)
            X_test_b = X_test_b.reset_index(drop=True)
            X_test_a = X_test_a.reset_index(drop=True)

            # Baseline model
            model_b = XGBRegressor(**best_xgb_params)
            model_b.fit(X_train_b, y_train)
            pred_b = current_values + model_b.predict(X_test_b)

            # Actor-enriched model
            model_a = XGBRegressor(**best_xgb_params)
            model_a.fit(X_train_a, y_train)
            pred_a = current_values + model_a.predict(X_test_a)

            y_true = current_values + y_test

            rmse_b.append(np.sqrt(mean_squared_error(y_true, pred_b)))
            mae_b.append(mean_absolute_error(y_true, pred_b))
            r2_b.append(r2_score(y_true, pred_b))

            rmse_a.append(np.sqrt(mean_squared_error(y_true, pred_a)))
            mae_a.append(mean_absolute_error(y_true, pred_a))
            r2_a.append(r2_score(y_true, pred_a))

            all_true.extend(y_true)
            all_pred_b.extend(pred_b)
            all_pred_a.extend(pred_a)
            all_err_b.extend(y_true - pred_b)
            all_err_a.extend(y_true - pred_a)

            imp_b.append(dict(zip(X_train_b.columns, model_b.feature_importances_)))
            imp_a.append(dict(zip(X_train_a.columns, model_a.feature_importances_)))

            # Save last-fold model and predictions
            if fold == tscv.get_n_splits() - 1:
                key = ("XGBoost", target_var, horizon)

                trained_models[(key, "baseline")] = model_b
                trained_models[(key, "actor")] = model_a

                cv_predictions[key] = {
                    "index": df_test_fe.index,
                    "y_true": y_true,
                    "y_pred_baseline": pred_b,
                    "y_pred_actor": pred_a
                }

                feature_sets[key] = {
                    "baseline_features": baseline_features,
                    "actor_features": actor_features,
                    "all_actor_model_features": baseline_features + actor_features
                }

        result_row = summarize_model_comparison(
            "XGBoost",
            target_var,
            horizon,
            rmse_b,
            rmse_a,
            mae_b,
            mae_a,
            r2_b,
            r2_a
        )

        all_results.append(result_row)

        print(pd.DataFrame([result_row]).to_markdown(index=False))

# -----------------------------
# Final CV summary
# -----------------------------
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(["Target", "Horizon"]).reset_index(drop=True)

print("\nFinal XGBoost CV Summary:")
print(results_df.to_markdown(index=False))

In [ ]:
# ============================================================
# Save and Download XGBoost CV Training Artifacts
# ============================================================

import os
import joblib

SAVE_DIR = "xgb_training_artifacts"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save summary table
results_df.to_csv(f"{SAVE_DIR}/xgb_cv_results.csv", index=False)

# Save trained last-fold CV models
joblib.dump(trained_models, f"{SAVE_DIR}/xgb_cv_trained_models.joblib")

# Save predictions
joblib.dump(cv_predictions, f"{SAVE_DIR}/xgb_cv_predictions.joblib")

# Save feature sets
joblib.dump(feature_sets, f"{SAVE_DIR}/xgb_feature_sets.joblib")

# Save hyperparameters
joblib.dump(best_xgb_params, f"{SAVE_DIR}/xgb_best_params.joblib")

# Save raw result object
joblib.dump(all_results, f"{SAVE_DIR}/xgb_all_results.joblib")

# Zip folder
!zip -r xgb_training_artifacts.zip xgb_training_artifacts

from google.colab import files
files.download("xgb_training_artifacts.zip")

In [ ]:
import joblib

trained_models = joblib.load("xgb_training_artifacts/xgb_cv_trained_models.joblib")
cv_predictions = joblib.load("xgb_training_artifacts/xgb_cv_predictions.joblib")
feature_sets = joblib.load("xgb_training_artifacts/xgb_feature_sets.joblib")
best_xgb_params = joblib.load("xgb_training_artifacts/xgb_best_params.joblib")

model_actor = trained_models[(("XGBoost", "Avg_Elapsed_Time", 1), "actor")]
model_baseline = trained_models[(("XGBoost", "Avg_Elapsed_Time", 1), "baseline")]

In [ ]:
# ============================================================
# Multi-Seed Final Holdout Evaluation + Statistical Tests
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import wilcoxon

print("\n🎯 Multi-seed final holdout evaluation...")

seeds = [1, 7, 13, 21, 42, 66, 77, 88, 99, 123]

multi_seed_holdout_results = []
holdout_predictions = {}

def fmt_mean_std(values):
    return f"{np.mean(values):.3f} ± {np.std(values, ddof=1):.3f}"

for horizon in horizons:
    for target_var in targets:
        print(f"\n==============================")
        print(f"Target: {target_var} | Horizon: {horizon}")
        print("==============================")

        # Feature engineering only needs to happen once per target/horizon
        df_train_fe, baseline_features, actor_features = feature_engineering(
            df_train_full,
            target_var=target_var,
            horizon=horizon
        )

        df_context = pd.concat([df_train_full.tail(40), df_test_final])
        df_test_fe, _, _ = feature_engineering(
            df_context,
            target_var=target_var,
            horizon=horizon
        )
        df_test_fe = df_test_fe.loc[df_test_final.index.intersection(df_test_fe.index)]

        df_train_fe.dropna(inplace=True)
        df_test_fe.dropna(inplace=True)

        X_train_b = df_train_fe[baseline_features]
        X_train_a = df_train_fe[baseline_features + actor_features]
        y_train = df_train_fe["target"]

        X_test_b = df_test_fe[baseline_features].reset_index(drop=True)
        X_test_a = df_test_fe[baseline_features + actor_features].reset_index(drop=True)
        y_test = df_test_fe["target"].reset_index(drop=True)
        current_values = df_test_fe[target_var].reset_index(drop=True)

        y_true = current_values + y_test

        for seed in seeds:
            print(f"Seed: {seed}")

            params = best_xgb_params.copy()
            params["random_state"] = seed

            # Baseline model
            final_model_b = XGBRegressor(**params)
            final_model_b.fit(X_train_b, y_train)
            pred_b = current_values + final_model_b.predict(X_test_b)

            # Actor-enriched model
            final_model_a = XGBRegressor(**params)
            final_model_a.fit(X_train_a, y_train)
            pred_a = current_values + final_model_a.predict(X_test_a)

            rmse_b = np.sqrt(mean_squared_error(y_true, pred_b))
            rmse_a = np.sqrt(mean_squared_error(y_true, pred_a))

            mae_b = mean_absolute_error(y_true, pred_b)
            mae_a = mean_absolute_error(y_true, pred_a)

            r2_b = r2_score(y_true, pred_b)
            r2_a = r2_score(y_true, pred_a)

            multi_seed_holdout_results.append({
                "Model": "XGBoost",
                "Target": target_var,
                "Horizon": horizon,
                "Seed": seed,
                "RMSE Baseline": rmse_b,
                "RMSE Actor": rmse_a,
                "Delta RMSE": rmse_b - rmse_a,
                "MAE Baseline": mae_b,
                "MAE Actor": mae_a,
                "Delta MAE": mae_b - mae_a,
                "R2 Baseline": r2_b,
                "R2 Actor": r2_a,
                "Delta R2": r2_a - r2_b
            })

            holdout_predictions[("XGBoost", target_var, horizon, seed)] = {
                "index": df_test_fe.index,
                "y_true": y_true,
                "pred_b": pred_b,
                "pred_a": pred_a,
                "model_b": final_model_b,
                "model_a": final_model_a,
                "X_test_b": X_test_b,
                "X_test_a": X_test_a,
                "baseline_features": baseline_features,
                "actor_features": actor_features
            }

multi_seed_df = pd.DataFrame(multi_seed_holdout_results)

print("\n📊 Multi-seed holdout results:")
print(multi_seed_df.to_markdown(index=False))

In [ ]:
# ============================================================
# Summary + Wilcoxon Statistical Tests
# ============================================================

stat_rows = []

for target_var in targets:
    for horizon in horizons:
        subset = multi_seed_df[
            (multi_seed_df["Target"] == target_var) &
            (multi_seed_df["Horizon"] == horizon)
        ].copy()

        rmse_b = subset["RMSE Baseline"].values
        rmse_a = subset["RMSE Actor"].values

        mae_b = subset["MAE Baseline"].values
        mae_a = subset["MAE Actor"].values

        r2_b = subset["R2 Baseline"].values
        r2_a = subset["R2 Actor"].values

        # One-sided Wilcoxon:
        # H1: baseline error > actor error
        stat_rmse, p_rmse = wilcoxon(rmse_b, rmse_a, alternative="greater")
        stat_mae, p_mae = wilcoxon(mae_b, mae_a, alternative="greater")

        stat_rows.append({
            "Model": "XGBoost",
            "Target": target_var,
            "Horizon": horizon,

            "RMSE Baseline": fmt_mean_std(rmse_b),
            "RMSE Actor": fmt_mean_std(rmse_a),
            "Delta RMSE": f"{np.mean(rmse_b - rmse_a):.3f}",
            "RMSE p-value": f"{p_rmse:.4f}",

            "MAE Baseline": fmt_mean_std(mae_b),
            "MAE Actor": fmt_mean_std(mae_a),
            "Delta MAE": f"{np.mean(mae_b - mae_a):.3f}",
            "MAE p-value": f"{p_mae:.4f}",

            "R2 Baseline": fmt_mean_std(r2_b),
            "R2 Actor": fmt_mean_std(r2_a),
            "Delta R2": f"{np.mean(r2_a - r2_b):.3f}"
        })

stats_df = pd.DataFrame(stat_rows)

print("\n📊 Multi-seed holdout summary with Wilcoxon tests:")
print(stats_df.to_markdown(index=False))

In [ ]:
# ============================================================
# Save Multi-Seed Holdout Results
# ============================================================

SAVE_DIR = "xgb_multi_seed_holdout"
os.makedirs(SAVE_DIR, exist_ok=True)

multi_seed_df.to_csv(f"{SAVE_DIR}/xgb_multi_seed_raw_results.csv", index=False)
stats_df.to_csv(f"{SAVE_DIR}/xgb_multi_seed_summary_stats.csv", index=False)

joblib.dump(holdout_predictions, f"{SAVE_DIR}/xgb_multi_seed_predictions_models.joblib")
joblib.dump(best_xgb_params, f"{SAVE_DIR}/xgb_best_params.joblib")

!zip -r xgb_multi_seed_holdout.zip xgb_multi_seed_holdout

from google.colab import files
files.download("xgb_multi_seed_holdout.zip")

In [ ]:
# ============================================================
# Aggregated SHAP for Multi-Seed XGBoost Holdout Models
# ============================================================

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

shap_rows = []

for horizon in horizons:
    for target_var in targets:
        for seed in seeds:

            key = ("XGBoost", target_var, horizon, seed)

            final_model_a = holdout_predictions[key]["model_a"]
            X_test_a = holdout_predictions[key]["X_test_a"].copy()

            print(f"SHAP | Target={target_var} | Horizon={horizon} | Seed={seed}")

            explainer_a = shap.Explainer(final_model_a, X_test_a)
            shap_values_a = explainer_a(X_test_a)

            mean_abs_shap = np.abs(shap_values_a.values).mean(axis=0)

            for feature, value in zip(X_test_a.columns, mean_abs_shap):
                shap_rows.append({
                    "Model": "XGBoost",
                    "Target": target_var,
                    "Horizon": horizon,
                    "Seed": seed,
                    "Feature": feature,
                    "MeanAbsSHAP": value
                })

shap_df = pd.DataFrame(shap_rows)

# Aggregate over seeds, targets, and horizons
shap_summary = (
    shap_df
    .groupby("Feature", as_index=False)["MeanAbsSHAP"]
    .mean()
    .sort_values("MeanAbsSHAP", ascending=False)
    .reset_index(drop=True)
)

print(shap_summary.head(20).to_markdown(index=False))

In [ ]:
top_k = 5

plot_df = shap_summary.head(top_k).sort_values("MeanAbsSHAP", ascending=True)

plt.figure(figsize=(8, 6))
plt.barh(plot_df["Feature"], plot_df["MeanAbsSHAP"])
plt.xlabel("Mean absolute SHAP value")
plt.ylabel("Feature")
plt.title("Aggregated SHAP Importance Across Seeds, Targets, and Horizons")
plt.tight_layout()
plt.savefig("xgb_aggregated_shap_importance.pdf", bbox_inches="tight")
plt.show()

In [ ]:
top_k = 10

plot_df = shap_summary.head(top_k).sort_values("MeanAbsSHAP", ascending=True)

plt.figure(figsize=(8, 6))
plt.barh(plot_df["Feature"], plot_df["MeanAbsSHAP"])
plt.xlabel("Mean absolute SHAP value")
plt.ylabel("Feature")
plt.title("Aggregated SHAP Importance Across Seeds, Targets, and Horizons")
plt.tight_layout()
plt.savefig("xgb_aggregated_shap_importance.pdf", bbox_inches="tight")
plt.show()

In [ ]:
def classify_feature_group(feature_name):
    actor_prefixes = [
        "Cum_Count_C", "Cum_Count_I", "Cum_Count_HI", "Cum_Count_HB",
        "Cum_Time_C_seconds", "Cum_Time_I_seconds",
        "Cum_Time_HI_seconds", "Cum_Time_HB_seconds"
    ]

    if any(feature_name.startswith(prefix) for prefix in actor_prefixes):
        return "Actor behavior"
    else:
        return "Process-performance / KPI"

shap_summary["Group"] = shap_summary["Feature"].apply(classify_feature_group)

group_summary = (
    shap_summary
    .groupby("Group", as_index=False)["MeanAbsSHAP"]
    .sum()
    .sort_values("MeanAbsSHAP", ascending=False)
)

print(group_summary.to_markdown(index=False))

plt.figure(figsize=(6, 4))
plt.bar(group_summary["Group"], group_summary["MeanAbsSHAP"])
plt.ylabel("Total aggregated SHAP importance")
plt.title("SHAP Contribution by Feature Group")
plt.tight_layout()
plt.savefig("xgb_shap_group_contribution.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# --- Final Holdout Evaluation for all targets ---
print("\n🎯 Testing on final unseen holdout set...")

from sklearn.utils import resample
import shap

def bootstrap_summary(y_true, y_pred, n_bootstrap=1000, seed=42):
    rng = np.random.RandomState(seed)
    rmse_vals, mae_vals, r2_vals = [], [], []
    n = len(y_true)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    for _ in range(n_bootstrap):
        idx = rng.choice(n, n, replace=True)
        yt_bs = y_true[idx]
        yp_bs = y_pred[idx]
        rmse_vals.append(np.sqrt(mean_squared_error(yt_bs, yp_bs)))
        mae_vals.append(mean_absolute_error(yt_bs, yp_bs))
        r2_vals.append(r2_score(yt_bs, yp_bs))

    def summary(vals):
        return np.mean(vals), np.std(vals)

    return {
        'rmse': summary(rmse_vals),
        'mae': summary(mae_vals),
        'r2': summary(r2_vals)
    }

def fmt(mean, std):
    return f"{mean:.3f} ± {std:.3f}"

for horizon in horizons:
  for target_var in targets:
      print(f"\n==============================")
      print(f"🎯 Final Holdout Target: {target_var}")
      print(f"==============================")

      # Feature engineering
      df_train_fe, baseline_features, actor_features = feature_engineering(
          df_train_full,
          target_var=target_var
      )

      df_context = pd.concat([df_train_full.tail(40), df_test_final])
      df_test_fe, _, _ = feature_engineering(
          df_context,
          target_var=target_var
      )
      df_test_fe = df_test_fe.loc[df_test_final.index.intersection(df_test_fe.index)]

      df_train_fe.dropna(inplace=True)
      df_test_fe.dropna(inplace=True)

      # Split features + target
      X_train_b = df_train_fe[baseline_features]
      X_train_a = df_train_fe[baseline_features + actor_features]
      y_train = df_train_fe["target"]

      X_test_b = df_test_fe[baseline_features]
      X_test_a = df_test_fe[baseline_features + actor_features]
      y_test = df_test_fe["target"].values

      # Train final models on full training data
      final_model_b = XGBRegressor(**best_xgb_params)
      final_model_b.fit(X_train_b, y_train)

      final_model_a = XGBRegressor(**best_xgb_params)
      final_model_a.fit(X_train_a, y_train)

      # Predict directly
      base_values = df_test_fe[target_var].shift(1).iloc[1:].reset_index(drop=True)

      y_test = pd.Series(y_test).iloc[1:].reset_index(drop=True)
      X_test_b = X_test_b.iloc[1:].reset_index(drop=True)
      X_test_a = X_test_a.iloc[1:].reset_index(drop=True)

      pred_b = final_model_b.predict(X_test_b) + base_values
      pred_a = final_model_a.predict(X_test_a) + base_values
      y_true = base_values + y_test

      # Bootstrap metrics
      metrics_b = bootstrap_summary(y_true, pred_b, seed=42)
      metrics_a = bootstrap_summary(y_true, pred_a, seed=42)

      # Point estimates and standard deviations
      mean_rmse_b, std_rmse_b = metrics_b["rmse"]
      mean_rmse_a, std_rmse_a = metrics_a["rmse"]
      mean_mae_b, std_mae_b = metrics_b["mae"]
      mean_mae_a, std_mae_a = metrics_a["mae"]
      mean_r2_b, std_r2_b = metrics_b["r2"]
      mean_r2_a, std_r2_a = metrics_a["r2"]

      # Plot predictions
      plot_index = df_test_fe.index[1:]

      plt.figure(figsize=(14, 6))
      plt.plot(plot_index, y_true, label='Actual', color='black')
      plt.plot(plot_index, pred_b, '--', label='XGBoost Baseline', color='green')
      plt.plot(plot_index, pred_a, '--', label='XGBoost Actor-Enriched', color='red')
      plt.title(f"📈 Final Holdout Test: {target_var}")
      plt.xlabel("Time")
      plt.ylabel(target_var)
      plt.legend()
      plt.grid(True)
      plt.tight_layout()
      plt.show()

      # Store summary row
      summary_row = {
          'Model': 'XGBoost',
          'Target': target_var,
          'Horizon': horizon,
          'RMSE Baseline': fmt(mean_rmse_b, std_rmse_b),
          'RMSE Actor': fmt(mean_rmse_a, std_rmse_a),
          'RMSE Δ': f"{mean_rmse_b - mean_rmse_a:.3f}",
          'MAE Baseline': fmt(mean_mae_b, std_mae_b),
          'MAE Actor': fmt(mean_mae_a, std_mae_a),
          'MAE Δ': f"{mean_mae_b - mean_mae_a:.3f}",
          'R² Baseline': fmt(mean_r2_b, std_r2_b),
          'R² Actor': fmt(mean_r2_a, std_r2_a),
          'R² Δ': f"{mean_r2_a - mean_r2_b:.3f}",
      }
      all_holdout_results.append(summary_row)

      # Store predictions/models for later TT bridge + SHAP
      holdout_predictions[target_var] = {
          "index": df_test_fe.index[1:],
          "y_true": y_true,
          "pred_b": pred_b,
          "pred_a": pred_a,
          "model_b": final_model_b,
          "model_a": final_model_a,
          "X_test_b": X_test_b,
          "X_test_a": X_test_a
      }

# Final summary across all 3 targets
summary_test = pd.DataFrame(all_holdout_results)
summary_test = summary_test.sort_values(["Target", "Horizon"]).reset_index(drop=True)

print("\n📊 Final Holdout Test Results Summary Across All Targets and Horizons:")
print(summary_test.to_markdown(index=False))

tuning

In [ ]:
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
import pandas as pd
import numpy as np
from itertools import product

targets = ["Avg_Elapsed_Time", "Avg_Remaining_Time", "WIP"]

# --- Define fixed hyperparameter sets to test ---
param_grid = {
    'n_estimators': [1000, 1500],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [3, 5, 6],
    'subsample': [0.9],
    'colsample_bytree': [0.9],
    'gamma': [0],
    'min_child_weight': [1]
}
param_combos = list(product(*param_grid.values()))
param_names = list(param_grid.keys())

all_holdout_tuning_results = []

for target_var in targets:
    print("\n" + "#" * 80)
    print(f"HOLDOUT TEST TUNING FOR TARGET: {target_var}")
    print("#" * 80)

    results_summary = []

    for i, param_values in enumerate(param_combos):
        params = dict(zip(param_names, param_values))

        print("\n" + "=" * 60)
        print(f"🔧 [{i+1}/{len(param_combos)}] Testing configuration for {target_var}:")
        for k, v in params.items():
            print(f"   {k}: {v}")
        print("=" * 60)

        # --------------------------------------------------
        # Build one context so holdout features use train history
        # --------------------------------------------------
        df_context = pd.concat([df_train_full, df_test_final], axis=0).copy()

        df_fe, baseline_features, actor_features = feature_engineering(
            df_context,
            target_var=target_var
        )

        # Split back into train / holdout using dates
        train_dates = set(df_train_full["date"])
        test_dates = set(df_test_final["date"])

        df_train_fe = df_fe[df_fe["date"].isin(train_dates)].copy()
        df_test_fe = df_fe[df_fe["date"].isin(test_dates)].copy()

        df_train_fe = df_train_fe.sort_values("date").reset_index(drop=True)
        df_test_fe = df_test_fe.sort_values("date").reset_index(drop=True)

        if df_train_fe.empty or df_test_fe.empty:
            print("   Skipping configuration: empty train/test after feature engineering")
            continue

        # Split features and target
        X_train_b = df_train_fe[baseline_features].copy()
        X_train_a = df_train_fe[baseline_features + actor_features].copy()
        y_train = df_train_fe["target"].copy()

        X_test_b = df_test_fe[baseline_features].copy()
        X_test_a = df_test_fe[baseline_features + actor_features].copy()
        y_test = df_test_fe["target"].copy()

        # Reconstruct KPI level from predicted smoothed change
        base_values = df_test_fe[target_var].shift(1)

        valid_mask = (
            X_test_b.notna().all(axis=1)
            & X_test_a.notna().all(axis=1)
            & y_test.notna()
            & base_values.notna()
        )

        X_test_b = X_test_b.loc[valid_mask].copy()
        X_test_a = X_test_a.loc[valid_mask].copy()
        y_test = y_test.loc[valid_mask].copy()
        base_values = base_values.loc[valid_mask].copy()
        plot_dates = df_test_fe.loc[valid_mask, "date"].copy().reset_index(drop=True)

        # Force numeric types
        X_train_b = X_train_b.apply(pd.to_numeric, errors="coerce").astype(float)
        X_train_a = X_train_a.apply(pd.to_numeric, errors="coerce").astype(float)
        X_test_b = X_test_b.apply(pd.to_numeric, errors="coerce").astype(float)
        X_test_a = X_test_a.apply(pd.to_numeric, errors="coerce").astype(float)
        y_train = pd.to_numeric(y_train, errors="coerce").astype(float)
        y_test = pd.to_numeric(y_test, errors="coerce").astype(float)
        base_values = pd.to_numeric(base_values, errors="coerce").astype(float)

        # Shared train rows so both models use same examples
        train_mask_shared = (
            X_train_b.notna().all(axis=1)
            & X_train_a.notna().all(axis=1)
            & y_train.notna()
        )

        X_train_b = X_train_b.loc[train_mask_shared].copy()
        X_train_a = X_train_a.loc[train_mask_shared].copy()
        y_train_final = y_train.loc[train_mask_shared].copy()

        if X_train_b.empty or X_test_b.empty or y_train_final.empty or y_test.empty:
            print("   Skipping configuration: empty data after alignment")
            continue

        # Baseline model
        model_b = XGBRegressor(**params, random_state=42)
        model_b.fit(X_train_b.to_numpy(), y_train_final.to_numpy())
        pred_b = model_b.predict(X_test_b.to_numpy()) + base_values.to_numpy()

        # Actor-enriched model
        model_a = XGBRegressor(**params, random_state=42)
        model_a.fit(X_train_a.to_numpy(), y_train_final.to_numpy())
        pred_a = model_a.predict(X_test_a.to_numpy()) + base_values.to_numpy()

        y_true = base_values.to_numpy() + y_test.to_numpy()

        rmse_b = np.sqrt(mean_squared_error(y_true, pred_b))
        rmse_a = np.sqrt(mean_squared_error(y_true, pred_a))
        delta = rmse_b - rmse_a
        outperforms = delta > 0

        print(
            f"✅ Holdout RMSE - Baseline: {rmse_b:.4f} | "
            f"Actor: {rmse_a:.4f} | "
            f"Δ = {delta:.4f} | "
            f"{'Actor Wins ✅' if outperforms else 'Baseline Wins ❌'}"
        )

        results_summary.append({
            'target': target_var,
            'params': params,
            'rmse_baseline_holdout': rmse_b,
            'rmse_actor_holdout': rmse_a,
            'delta_holdout': delta,
            'outperforms_holdout': outperforms
        })

    summary_df = pd.DataFrame(results_summary).sort_values(by='delta_holdout', ascending=False)

    print(f"\n\n📋 Final HOLDOUT Results Summary for {target_var}:")
    print(summary_df[['params', 'rmse_baseline_holdout', 'rmse_actor_holdout', 'delta_holdout', 'outperforms_holdout']].to_markdown(index=False))

    print(f"\n✅ Configurations where Actor-Enriched outperformed Baseline on HOLDOUT for {target_var}:")
    winners_df = summary_df[summary_df['outperforms_holdout']]
    if len(winners_df) > 0:
        print(winners_df[['params', 'rmse_baseline_holdout', 'rmse_actor_holdout', 'delta_holdout']].to_markdown(index=False))
    else:
        print("None")

    all_holdout_tuning_results.append(summary_df)

# --- Combined summary across all targets ---
all_holdout_tuning_results_df = pd.concat(all_holdout_tuning_results, ignore_index=True)

print("\n\n📋 Combined HOLDOUT Tuning Results Across Targets:")
print(all_holdout_tuning_results_df[['target', 'params', 'rmse_baseline_holdout', 'rmse_actor_holdout', 'delta_holdout', 'outperforms_holdout']].to_markdown(index=False))

print("\n\n✅ ONLY the configurations where Actor-Enriched outperformed Baseline on HOLDOUT:")
final_winners = all_holdout_tuning_results_df[all_holdout_tuning_results_df['outperforms_holdout']].copy()

if len(final_winners) > 0:
    print(final_winners[['target', 'params', 'rmse_baseline_holdout', 'rmse_actor_holdout', 'delta_holdout']].to_markdown(index=False))
else:
    print("None")

# LightGBM

In [ ]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel, sem, t, wilcoxon

In [ ]:
# Best LightGBM parameters from sweep
best_lgbm_params = {
    'n_estimators': 1500,
    'learning_rate': 0.05,
    'max_depth': 5,
    'bagging_fraction': 0.9,
    'feature_fraction': 0.9,
    'reg_alpha': 0,
    'reg_lambda': 0,
    'min_child_samples': 1,
    'random_state': 42
}

In [ ]:
horizons = [1, 3, 7]
targets = ["Avg_Elapsed_Time", "Avg_Remaining_Time"]

all_results = []
all_holdout_results = []
holdout_predictions = {}

In [ ]:
# LightGBM.py

# ============================================================
# LightGBM Training with Artifact Saving
# ============================================================

import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from lightgbm import LGBMRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import ttest_rel, sem, t, wilcoxon

SAVE_DIR = "lgbm_training_artifacts"
os.makedirs(SAVE_DIR, exist_ok=True)

best_lgbm_params = {
    'n_estimators': 1500,
    'learning_rate': 0.05,
    'max_depth': 5,
    'bagging_fraction': 0.9,
    'feature_fraction': 0.9,
    'reg_alpha': 0,
    'reg_lambda': 0,
    'min_child_samples': 1,
    'random_state': 42
}

horizons = [1, 3, 7]
targets = ["Avg_Elapsed_Time", "Avg_Remaining_Time"]

all_results = []
trained_models = {}
cv_predictions = {}
feature_sets = {}

def cohen_d(x, y):
    diff = np.array(x) - np.array(y)
    return diff.mean() / diff.std(ddof=1)

def confidence_interval(data, confidence=0.95):
    m = np.mean(data)
    se_val = sem(data)
    h = se_val * t.ppf((1 + confidence) / 2.0, len(data) - 1)
    return f"{m:.3f} ± {h:.3f}"

def summarize_model_comparison(name, target_name, horizon, rmse_b, rmse_a, mae_b, mae_a, r2_b, r2_a):
    stat, p_rmse = ttest_rel(rmse_b, rmse_a)
    stat_w, p_wilcoxon = wilcoxon(rmse_b, rmse_a)

    return {
        "Model": name,
        "Target": target_name,
        "Horizon": horizon,
        "RMSE Baseline": confidence_interval(rmse_b),
        "RMSE Actor": confidence_interval(rmse_a),
        "RMSE Δ": f"{np.mean(rmse_b) - np.mean(rmse_a):.3f}",
        "MAE Baseline": confidence_interval(mae_b),
        "MAE Actor": confidence_interval(mae_a),
        "MAE Δ": f"{np.mean(mae_b) - np.mean(mae_a):.3f}",
        "R² Baseline": confidence_interval(r2_b),
        "R² Actor": confidence_interval(r2_a),
        "R² Δ": f"{np.mean(r2_a) - np.mean(r2_b):.3f}",
        "p-value RMSE t-test": f"{p_rmse:.4f}",
        "p-value RMSE Wilcoxon": f"{p_wilcoxon:.4f}",
        "Cohen’s d": f"{cohen_d(rmse_b, rmse_a):.3f}"
    }

for horizon in horizons:
    for target_var in targets:
        print("\n==============================")
        print(f"Running LightGBM | Target: {target_var} | Horizon: {horizon}")
        print("==============================")

        df = df_train_full.copy()
        tscv = TimeSeriesSplit(n_splits=5)

        rmse_b, mae_b, r2_b = [], [], []
        rmse_a, mae_a, r2_a = [], [], []

        for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
            print(f"Fold {fold + 1}")

            df_train_raw = df.iloc[:test_idx[0]].copy()
            df_test_raw = df.iloc[test_idx].copy()

            df_train_fe, baseline_features, actor_features = feature_engineering(
                df_train_raw,
                target_var=target_var,
                horizon=horizon
            )

            df_context = pd.concat([df_train_raw.tail(40), df_test_raw])
            df_test_fe, _, _ = feature_engineering(
                df_context,
                target_var=target_var,
                horizon=horizon
            )
            df_test_fe = df_test_fe.loc[df_test_raw.index.intersection(df_test_fe.index)]

            df_train_fe.dropna(inplace=True)
            df_test_fe.dropna(inplace=True)

            X_train_b = df_train_fe[baseline_features]
            X_test_b = df_test_fe[baseline_features].reset_index(drop=True)

            X_train_a = df_train_fe[baseline_features + actor_features]
            X_test_a = df_test_fe[baseline_features + actor_features].reset_index(drop=True)

            y_train = df_train_fe["target"]
            y_test = df_test_fe["target"].reset_index(drop=True)

            current_values = df_test_fe[target_var].reset_index(drop=True)
            y_true = current_values + y_test

            model_b = LGBMRegressor(**best_lgbm_params)
            model_b.fit(X_train_b, y_train)
            pred_b = current_values + model_b.predict(X_test_b)

            model_a = LGBMRegressor(**best_lgbm_params)
            model_a.fit(X_train_a, y_train)
            pred_a = current_values + model_a.predict(X_test_a)

            rmse_b.append(np.sqrt(mean_squared_error(y_true, pred_b)))
            mae_b.append(mean_absolute_error(y_true, pred_b))
            r2_b.append(r2_score(y_true, pred_b))

            rmse_a.append(np.sqrt(mean_squared_error(y_true, pred_a)))
            mae_a.append(mean_absolute_error(y_true, pred_a))
            r2_a.append(r2_score(y_true, pred_a))

            if fold == tscv.get_n_splits() - 1:
                key = ("LightGBM", target_var, horizon)

                trained_models[(key, "baseline")] = model_b
                trained_models[(key, "actor")] = model_a

                cv_predictions[key] = {
                    "index": df_test_fe.index,
                    "y_true": y_true,
                    "y_pred_baseline": pred_b,
                    "y_pred_actor": pred_a
                }

                feature_sets[key] = {
                    "baseline_features": baseline_features,
                    "actor_features": actor_features,
                    "all_actor_model_features": baseline_features + actor_features
                }

        result_row = summarize_model_comparison(
            "LightGBM",
            target_var,
            horizon,
            rmse_b,
            rmse_a,
            mae_b,
            mae_a,
            r2_b,
            r2_a
        )

        all_results.append(result_row)
        print(pd.DataFrame([result_row]).to_markdown(index=False))

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(["Target", "Horizon"]).reset_index(drop=True)

print("\nFinal LightGBM CV Summary:")
print(results_df.to_markdown(index=False))

results_df.to_csv(f"{SAVE_DIR}/lgbm_cv_results.csv", index=False)
joblib.dump(trained_models, f"{SAVE_DIR}/lgbm_cv_trained_models.joblib")
joblib.dump(cv_predictions, f"{SAVE_DIR}/lgbm_cv_predictions.joblib")
joblib.dump(feature_sets, f"{SAVE_DIR}/lgbm_feature_sets.joblib")
joblib.dump(best_lgbm_params, f"{SAVE_DIR}/lgbm_best_params.joblib")
joblib.dump(all_results, f"{SAVE_DIR}/lgbm_all_results.joblib")

!zip -r lgbm_training_artifacts.zip lgbm_training_artifacts

from google.colab import files
files.download("lgbm_training_artifacts.zip")

In [ ]:
# ============================================================
# Multi-Seed Final Holdout Evaluation + Statistical Tests
# LightGBM
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import wilcoxon

print("\n🎯 Multi-seed final holdout evaluation for LightGBM...")

seeds = [1, 7, 13, 21, 42, 66, 77, 88, 99, 123]

multi_seed_holdout_results = []
holdout_predictions = {}

def fmt_mean_std(values):
    return f"{np.mean(values):.3f} ± {np.std(values, ddof=1):.3f}"

for horizon in horizons:
    for target_var in targets:
        print(f"\n==============================")
        print(f"LightGBM | Target: {target_var} | Horizon: {horizon}")
        print("==============================")

        df_train_fe, baseline_features, actor_features = feature_engineering(
            df_train_full,
            target_var=target_var,
            horizon=horizon
        )

        df_context = pd.concat([df_train_full.tail(40), df_test_final])
        df_test_fe, _, _ = feature_engineering(
            df_context,
            target_var=target_var,
            horizon=horizon
        )
        df_test_fe = df_test_fe.loc[df_test_final.index.intersection(df_test_fe.index)]

        df_train_fe.dropna(inplace=True)
        df_test_fe.dropna(inplace=True)

        X_train_b = df_train_fe[baseline_features]
        X_train_a = df_train_fe[baseline_features + actor_features]
        y_train = df_train_fe["target"]

        X_test_b = df_test_fe[baseline_features].reset_index(drop=True)
        X_test_a = df_test_fe[baseline_features + actor_features].reset_index(drop=True)
        y_test = df_test_fe["target"].reset_index(drop=True)
        current_values = df_test_fe[target_var].reset_index(drop=True)

        y_true = current_values + y_test

        for seed in seeds:
            print(f"Seed: {seed}")

            params = best_lgbm_params.copy()
            params["random_state"] = seed

            model_b = LGBMRegressor(**params)
            model_b.fit(X_train_b, y_train)
            pred_b = current_values + model_b.predict(X_test_b)

            model_a = LGBMRegressor(**params)
            model_a.fit(X_train_a, y_train)
            pred_a = current_values + model_a.predict(X_test_a)

            rmse_b = np.sqrt(mean_squared_error(y_true, pred_b))
            rmse_a = np.sqrt(mean_squared_error(y_true, pred_a))

            mae_b = mean_absolute_error(y_true, pred_b)
            mae_a = mean_absolute_error(y_true, pred_a)

            r2_b = r2_score(y_true, pred_b)
            r2_a = r2_score(y_true, pred_a)

            multi_seed_holdout_results.append({
                "Model": "LightGBM",
                "Target": target_var,
                "Horizon": horizon,
                "Seed": seed,
                "RMSE Baseline": rmse_b,
                "RMSE Actor": rmse_a,
                "Delta RMSE": rmse_b - rmse_a,
                "MAE Baseline": mae_b,
                "MAE Actor": mae_a,
                "Delta MAE": mae_b - mae_a,
                "R2 Baseline": r2_b,
                "R2 Actor": r2_a,
                "Delta R2": r2_a - r2_b
            })

            holdout_predictions[("LightGBM", target_var, horizon, seed)] = {
                "index": df_test_fe.index,
                "y_true": y_true,
                "pred_b": pred_b,
                "pred_a": pred_a,
                "model_b": model_b,
                "model_a": model_a,
                "X_test_b": X_test_b,
                "X_test_a": X_test_a,
                "baseline_features": baseline_features,
                "actor_features": actor_features
            }

multi_seed_df = pd.DataFrame(multi_seed_holdout_results)

print("\n📊 Multi-seed holdout raw results:")
print(multi_seed_df.to_markdown(index=False))

stat_rows = []

for target_var in targets:
    for horizon in horizons:
        subset = multi_seed_df[
            (multi_seed_df["Target"] == target_var) &
            (multi_seed_df["Horizon"] == horizon)
        ].copy()

        rmse_b = subset["RMSE Baseline"].values
        rmse_a = subset["RMSE Actor"].values

        mae_b = subset["MAE Baseline"].values
        mae_a = subset["MAE Actor"].values

        r2_b = subset["R2 Baseline"].values
        r2_a = subset["R2 Actor"].values

        stat_rmse, p_rmse = wilcoxon(rmse_b, rmse_a, alternative="greater")
        stat_mae, p_mae = wilcoxon(mae_b, mae_a, alternative="greater")

        stat_rows.append({
            "Model": "LightGBM",
            "Target": target_var,
            "Horizon": horizon,

            "RMSE Baseline": fmt_mean_std(rmse_b),
            "RMSE Actor": fmt_mean_std(rmse_a),
            "Delta RMSE": f"{np.mean(rmse_b - rmse_a):.3f}",
            "RMSE p-value": f"{p_rmse:.4f}",

            "MAE Baseline": fmt_mean_std(mae_b),
            "MAE Actor": fmt_mean_std(mae_a),
            "Delta MAE": f"{np.mean(mae_b - mae_a):.3f}",
            "MAE p-value": f"{p_mae:.4f}",

            "R2 Baseline": fmt_mean_std(r2_b),
            "R2 Actor": fmt_mean_std(r2_a),
            "Delta R2": f"{np.mean(r2_a - r2_b):.3f}"
        })

stats_df = pd.DataFrame(stat_rows)

print("\n📊 Multi-seed holdout summary with Wilcoxon tests:")
print(stats_df.to_markdown(index=False))

SAVE_DIR = "lgbm_multi_seed_holdout"
os.makedirs(SAVE_DIR, exist_ok=True)

multi_seed_df.to_csv(f"{SAVE_DIR}/lgbm_multi_seed_raw_results.csv", index=False)
stats_df.to_csv(f"{SAVE_DIR}/lgbm_multi_seed_summary_stats.csv", index=False)

joblib.dump(holdout_predictions, f"{SAVE_DIR}/lgbm_multi_seed_predictions_models.joblib")
joblib.dump(best_lgbm_params, f"{SAVE_DIR}/lgbm_best_params.joblib")

!zip -r lgbm_multi_seed_holdout.zip lgbm_multi_seed_holdout

from google.colab import files
files.download("lgbm_multi_seed_holdout.zip")

In [ ]:
# ============================================================
# Aggregated SHAP for Multi-Seed LightGBM Holdout Models
# ============================================================

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

shap_rows = []

for horizon in horizons:
    for target_var in targets:
        for seed in seeds:
            key = ("LightGBM", target_var, horizon, seed)

            model_a = holdout_predictions[key]["model_a"]
            X_test_a = holdout_predictions[key]["X_test_a"].copy()

            print(f"SHAP | LightGBM | Target={target_var} | Horizon={horizon} | Seed={seed}")

            explainer = shap.Explainer(model_a, X_test_a)
            shap_values = explainer(X_test_a, check_additivity=False)

            mean_abs_shap = np.abs(shap_values.values).mean(axis=0)

            for feature, value in zip(X_test_a.columns, mean_abs_shap):
                shap_rows.append({
                    "Model": "LightGBM",
                    "Target": target_var,
                    "Horizon": horizon,
                    "Seed": seed,
                    "Feature": feature,
                    "MeanAbsSHAP": value
                })

shap_df = pd.DataFrame(shap_rows)

shap_summary = (
    shap_df
    .groupby("Feature", as_index=False)["MeanAbsSHAP"]
    .mean()
    .sort_values("MeanAbsSHAP", ascending=False)
    .reset_index(drop=True)
)

print("\nTop aggregated SHAP features:")
print(shap_summary.head(20).to_markdown(index=False))

top_k = 10
plot_df = shap_summary.head(top_k).sort_values("MeanAbsSHAP", ascending=True)

plt.figure(figsize=(8, 6))
plt.barh(plot_df["Feature"], plot_df["MeanAbsSHAP"])
plt.xlabel("Mean absolute SHAP value")
plt.ylabel("Feature")
plt.title("LightGBM Aggregated SHAP Importance")
plt.tight_layout()
plt.savefig("lgbm_aggregated_shap_importance.pdf", bbox_inches="tight")
plt.show()

def classify_feature_group(feature_name):
    actor_prefixes = [
        "Cum_Count_C", "Cum_Count_I", "Cum_Count_HI", "Cum_Count_HB",
        "Cum_Time_C_seconds", "Cum_Time_I_seconds",
        "Cum_Time_HI_seconds", "Cum_Time_HB_seconds"
    ]

    if any(feature_name.startswith(prefix) for prefix in actor_prefixes):
        return "Actor behavior"
    else:
        return "Process-performance / KPI"

shap_summary["Group"] = shap_summary["Feature"].apply(classify_feature_group)

group_summary = (
    shap_summary
    .groupby("Group", as_index=False)["MeanAbsSHAP"]
    .sum()
    .sort_values("MeanAbsSHAP", ascending=False)
)

print("\nSHAP contribution by feature group:")
print(group_summary.to_markdown(index=False))

plt.figure(figsize=(6, 4))
plt.bar(group_summary["Group"], group_summary["MeanAbsSHAP"])
plt.ylabel("Total aggregated SHAP importance")
plt.title("LightGBM SHAP Contribution by Feature Group")
plt.tight_layout()
plt.savefig("lgbm_shap_group_contribution.pdf", bbox_inches="tight")
plt.show()

SAVE_DIR = "lgbm_shap_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

shap_df.to_csv(f"{SAVE_DIR}/lgbm_shap_raw.csv", index=False)
shap_summary.to_csv(f"{SAVE_DIR}/lgbm_shap_summary.csv", index=False)
group_summary.to_csv(f"{SAVE_DIR}/lgbm_shap_group_summary.csv", index=False)

!zip -r lgbm_shap_outputs.zip lgbm_shap_outputs

from google.colab import files
files.download("lgbm_shap_outputs.zip")

tuning

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import pandas as pd
import numpy as np
from itertools import product

# --- Define LightGBM hyperparameter sets to test ---
param_grid = {
    'n_estimators': [1000, 1500],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [3, 5, 6],
    'subsample': [0.9],             # LightGBM: bagging_fraction
    'colsample_bytree': [0.9],      # LightGBM: feature_fraction
    'reg_alpha': [0],               # L1 regularization
    'reg_lambda': [0],              # L2 regularization
    'min_child_weight': [1]         # LightGBM: min_child_samples (mapped below)
}
param_combos = list(product(*param_grid.values()))
param_names = list(param_grid.keys())

# --- Load data ---
df = df_train_full.copy()
tscv = TimeSeriesSplit(n_splits=5)

results_summary = []

# --- Loop through hyperparameter combinations ---
for i, param_values in enumerate(param_combos):
    params_raw = dict(zip(param_names, param_values))

    # Map params to LightGBM equivalents
    params = {
        'n_estimators': params_raw['n_estimators'],
        'learning_rate': params_raw['learning_rate'],
        'max_depth': params_raw['max_depth'],
        'bagging_fraction': params_raw['subsample'],
        'feature_fraction': params_raw['colsample_bytree'],
        'reg_alpha': params_raw['reg_alpha'],
        'reg_lambda': params_raw['reg_lambda'],
        'min_child_samples': params_raw['min_child_weight'],
        'random_state': 42
    }

    print("\n" + "="*60)
    print(f"🔧 [{i+1}/{len(param_combos)}] Testing configuration:")
    for k, v in params.items():
        print(f"   {k}: {v}")
    print("="*60)

    rmse_b, rmse_a = [], []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
        df_train_raw = df.iloc[:test_idx[0]].copy()
        df_test_raw = df.iloc[test_idx].copy()

        df_train_fe, baseline_features, actor_features = feature_engineering(df_train_raw, residual_target=True)
        df_context = pd.concat([df_train_raw.tail(40), df_test_raw])
        df_test_fe, _, _ = feature_engineering(df_context, residual_target=True)
        df_test_fe = df_test_fe.loc[df_test_raw.index.intersection(df_test_fe.index)]

        df_train_fe.dropna(inplace=True)
        df_test_fe.dropna(inplace=True)

        X_train_b = df_train_fe[baseline_features]
        X_test_b = df_test_fe[baseline_features]
        X_train_a = df_train_fe[baseline_features + actor_features]
        X_test_a = df_test_fe[baseline_features + actor_features]
        y_train = df_train_fe['target']
        y_test = df_test_fe['target']

        base_values = df_test_fe['TT'].shift(1).iloc[1:].reset_index(drop=True)
        y_test = y_test.iloc[1:].reset_index(drop=True)
        X_test_b = X_test_b.iloc[1:].reset_index(drop=True)
        X_test_a = X_test_a.iloc[1:].reset_index(drop=True)

        model_b = LGBMRegressor(**params)
        model_b.fit(X_train_b, y_train)
        pred_b = model_b.predict(X_test_b) + base_values

        model_a = LGBMRegressor(**params)
        model_a.fit(X_train_a, y_train)
        pred_a = model_a.predict(X_test_a) + base_values

        y_true = base_values + y_test

        rmse_fold_b = np.sqrt(mean_squared_error(y_true, pred_b))
        rmse_fold_a = np.sqrt(mean_squared_error(y_true, pred_a))
        rmse_b.append(rmse_fold_b)
        rmse_a.append(rmse_fold_a)

        print(f"   📉 Fold {fold+1} RMSE - Baseline: {rmse_fold_b:.4f} | Actor: {rmse_fold_a:.4f}")

    avg_rmse_b = np.mean(rmse_b)
    avg_rmse_a = np.mean(rmse_a)
    delta = avg_rmse_b - avg_rmse_a
    outperforms = delta > 0

    print(f"✅ Avg RMSE - Baseline: {avg_rmse_b:.4f} | Actor: {avg_rmse_a:.4f} | Δ = {delta:.4f} | {'Actor Wins ✅' if outperforms else 'Baseline Wins ❌'}")

    results_summary.append({
        'params': params,
        'rmse_baseline': avg_rmse_b,
        'rmse_actor': avg_rmse_a,
        'delta': delta,
        'outperforms': outperforms
    })

# --- Final Summary ---
summary_df = pd.DataFrame(results_summary)
summary_df = summary_df.sort_values(by='delta', ascending=False)

print("\n\n📋 Final Results Summary:")
print(summary_df[['params', 'rmse_baseline', 'rmse_actor', 'delta', 'outperforms']].to_markdown(index=False))

print("\n✅ Configurations where Actor-Enriched outperformed Baseline:")
print(summary_df[summary_df['outperforms']][['params', 'rmse_baseline', 'rmse_actor', 'delta']].to_markdown(index=False))


In [ ]:
print("\n✅ Configurations where Actor-Enriched outperformed Baseline:")
print(summary_df[summary_df['outperforms']][['params', 'rmse_baseline', 'rmse_actor', 'delta']].to_markdown(index=False))


# RNN

In [ ]:
# ============================================================
# RNN WITH ATTENTION: Imports, Model, Helpers
# ============================================================

import os
import gc
import random
import time
import logging
import zipfile
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tqdm.auto import tqdm

from tensorflow.keras.layers import (
    LSTM, GRU, Dense, Dropout, Input, Bidirectional,
    Conv1D, MaxPooling1D, GlobalAveragePooling1D,
    LayerNormalization, MultiHeadAttention
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import ttest_rel, sem, t, wilcoxon

logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.get_logger().setLevel("ERROR")


# ============================================================
# Reproducibility
# ============================================================

def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    tf.keras.utils.set_random_seed(seed)

    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


# ============================================================
# Sequence Data
# ============================================================

def create_seq_data(df, features, time_steps=15, target_col="target_scaled"):
    X, y, idx = [], [], []

    for i in range(time_steps, len(df)):
        X.append(df[features].iloc[i - time_steps:i].values)
        y.append(df[target_col].iloc[i])
        idx.append(df.index[i])

    return np.array(X), np.array(y), idx


# ============================================================
# Train / Validation Split
# ============================================================

def make_train_val_split(X_train, y_train, val_fraction=0.15):
    """
    Time-series-safe validation split.
    Uses the final part of the training data as validation.
    Does NOT touch the test/holdout set.
    """
    val_size = max(int(val_fraction * len(X_train)), 1)

    if len(X_train) <= val_size:
        raise ValueError(
            f"Not enough training sequences for validation split: "
            f"len(X_train)={len(X_train)}, val_size={val_size}"
        )

    X_fit = X_train[:-val_size]
    y_fit = y_train[:-val_size]

    X_val = X_train[-val_size:]
    y_val = y_train[-val_size:]

    return X_fit, y_fit, X_val, y_val


# ============================================================
# Statistics Helpers
# ============================================================

def cohen_d(x, y):
    diff = np.array(x) - np.array(y)
    std = diff.std(ddof=1)

    if std == 0:
        return np.nan

    return diff.mean() / std


def confidence_interval(data, confidence=0.95):
    data = np.asarray(data)
    data = data[~np.isnan(data)]

    if len(data) == 0:
        return "NA"

    if len(data) == 1:
        return f"{data[0]:.3f} ± NA"

    m = np.mean(data)
    se = sem(data)
    h = se * t.ppf((1 + confidence) / 2.0, len(data) - 1)

    return f"{m:.3f} ± {h:.3f}"


def fmt_mean_std(values):
    values = np.asarray(values)
    values = values[~np.isnan(values)]

    if len(values) == 0:
        return "NA"

    if len(values) == 1:
        return f"{values[0]:.3f} ± NA"

    return f"{np.mean(values):.3f} ± {np.std(values, ddof=1):.3f}"


def summarize_model_comparison(
    name,
    target_name,
    horizon,
    rmse_b,
    rmse_a,
    mae_b,
    mae_a,
    r2_b,
    r2_a
):
    rmse_b = np.asarray(rmse_b)
    rmse_a = np.asarray(rmse_a)

    mae_b = np.asarray(mae_b)
    mae_a = np.asarray(mae_a)

    r2_b = np.asarray(r2_b)
    r2_a = np.asarray(r2_a)

    try:
        _, p_rmse_t = ttest_rel(rmse_b, rmse_a)
    except Exception:
        p_rmse_t = np.nan

    try:
        _, p_rmse_w = wilcoxon(rmse_b, rmse_a)
    except Exception:
        p_rmse_w = np.nan

    return {
        "Model": name,
        "Target": target_name,
        "Horizon": horizon,

        "RMSE Baseline": confidence_interval(rmse_b),
        "RMSE Actor": confidence_interval(rmse_a),
        "RMSE Delta": f"{np.mean(rmse_b - rmse_a):.3f}",

        "MAE Baseline": confidence_interval(mae_b),
        "MAE Actor": confidence_interval(mae_a),
        "MAE Delta": f"{np.mean(mae_b - mae_a):.3f}",

        "R2 Baseline": confidence_interval(r2_b),
        "R2 Actor": confidence_interval(r2_a),
        "R2 Delta": f"{np.mean(r2_a - r2_b):.3f}",

        "p-value RMSE t-test": f"{p_rmse_t:.4f}" if not np.isnan(p_rmse_t) else "NA",
        "p-value RMSE Wilcoxon": f"{p_rmse_w:.4f}" if not np.isnan(p_rmse_w) else "NA",
        "Cohen d RMSE": f"{cohen_d(rmse_b, rmse_a):.3f}",
    }


# ============================================================
# Block Bootstrap Helpers
# ============================================================

def block_bootstrap_pvalue(delta, block_size=10, n_boot=5000, seed=42):
    """
    One-sided block bootstrap.

    delta = error_baseline - error_actor

    Positive delta means actor is better.
    p-value = probability that the mean improvement is <= 0.
    """
    rng = np.random.default_rng(seed)

    delta = np.asarray(delta)
    delta = delta[~np.isnan(delta)]

    n = len(delta)

    if n == 0:
        return np.nan, np.nan

    if block_size > n:
        block_size = n

    observed = np.mean(delta)

    boot_means = []

    for _ in range(n_boot):
        sampled = []

        while len(sampled) < n:
            start = rng.integers(0, n - block_size + 1)
            sampled.extend(delta[start:start + block_size])

        sampled = np.asarray(sampled[:n])
        boot_means.append(np.mean(sampled))

    boot_means = np.asarray(boot_means)

    p_value = np.mean(boot_means <= 0)

    return observed, p_value


def paired_error_bootstrap_tests(
    preds_b,
    preds_a,
    block_size=10,
    n_boot=5000,
    seed=42
):
    """
    Per-observation paired error test.

    Compares:
    - squared error baseline vs actor
    - absolute error baseline vs actor
    """
    y_true_b = np.asarray(preds_b["y_true"])
    y_true_a = np.asarray(preds_a["y_true"])

    pred_b = np.asarray(preds_b["y_pred"])
    pred_a = np.asarray(preds_a["y_pred"])

    if len(y_true_b) != len(y_true_a):
        raise ValueError("Baseline and actor prediction lengths differ.")

    if not np.allclose(y_true_b, y_true_a, equal_nan=True):
        print("Warning: y_true differs between baseline and actor. Using baseline y_true.")

    y_true = y_true_b

    squared_error_b = (y_true - pred_b) ** 2
    squared_error_a = (y_true - pred_a) ** 2

    absolute_error_b = np.abs(y_true - pred_b)
    absolute_error_a = np.abs(y_true - pred_a)

    delta_se = squared_error_b - squared_error_a
    delta_ae = absolute_error_b - absolute_error_a

    mean_delta_se, p_se = block_bootstrap_pvalue(
        delta_se,
        block_size=block_size,
        n_boot=n_boot,
        seed=seed
    )

    mean_delta_ae, p_ae = block_bootstrap_pvalue(
        delta_ae,
        block_size=block_size,
        n_boot=n_boot,
        seed=seed
    )

    return {
        "Mean Delta Squared Error": mean_delta_se,
        "Squared Error Bootstrap p-value": p_se,

        "Mean Delta Absolute Error": mean_delta_ae,
        "Absolute Error Bootstrap p-value": p_ae,

        "N Observations": len(y_true)
    }


# ============================================================
# Model
# ============================================================

def build_attention_rnn(input_shape, rnn_type="gru"):
    input_layer = Input(shape=input_shape)

    x = Conv1D(
        filters=64,
        kernel_size=3,
        activation="relu",
        padding="same"
    )(input_layer)

    x = MaxPooling1D(pool_size=2)(x)

    if rnn_type == "lstm":
        x = Bidirectional(LSTM(64, return_sequences=True))(x)
        x = Dropout(0.2)(x)
        x = LSTM(32, return_sequences=True)(x)
    else:
        x = Bidirectional(GRU(64, return_sequences=True))(x)
        x = Dropout(0.2)(x)
        x = GRU(32, return_sequences=True)(x)

    x = Dropout(0.2)(x)

    attn = MultiHeadAttention(
        num_heads=4,
        key_dim=16
    )(x, x)

    x = LayerNormalization(epsilon=1e-6)(x + attn)

    x = GlobalAveragePooling1D()(x)

    output = Dense(1)(x)

    model = Model(inputs=input_layer, outputs=output)

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    return model

In [ ]:
# ============================================================
# RNN WITH ATTENTION: Cross-Validation Training
# ============================================================

def train_rnn_cv_for_target_horizon(
    df_train_full,
    target_var,
    horizon,
    feature_set="baseline",
    rnn_type="gru",
    time_steps=15,
    n_splits=5,
    seed=42
):
    set_seed(seed)

    CONTEXT_ROWS = 200
    tscv = TimeSeriesSplit(n_splits=n_splits)

    rmse_list, mae_list, r2_list = [], [], []
    final_preds = None

    fold_iterator = tqdm(
        enumerate(tscv.split(df_train_full)),
        total=n_splits,
        desc=f"{rnn_type.upper()} {feature_set} | {target_var} | H={horizon}",
        leave=False
    )

    for fold, (train_idx, test_idx) in fold_iterator:
        df_train_raw = df_train_full.iloc[train_idx].copy()
        df_test_raw = df_train_full.iloc[test_idx].copy()

        df_train_fe, baseline_features, actor_features = feature_engineering(
            df_train_raw,
            target_var=target_var,
            horizon=horizon
        )

        df_context = pd.concat([
            df_train_raw.tail(CONTEXT_ROWS),
            df_test_raw
        ])

        df_test_fe, _, _ = feature_engineering(
            df_context,
            target_var=target_var,
            horizon=horizon
        )

        df_test_fe = df_test_fe.loc[
            df_test_raw.index.intersection(df_test_fe.index)
        ]

        df_train_fe.dropna(inplace=True)
        df_test_fe.dropna(inplace=True)

        df_train_fe = df_train_fe.copy()
        df_test_fe = df_test_fe.copy()

        if feature_set == "baseline":
            features = baseline_features
        else:
            features = baseline_features + actor_features

        scaler_x = StandardScaler()
        scaler_y = StandardScaler()

        df_train_fe[features] = scaler_x.fit_transform(df_train_fe[features])
        df_test_fe[features] = scaler_x.transform(df_test_fe[features])

        df_train_fe["target_scaled"] = scaler_y.fit_transform(
            df_train_fe[["target"]]
        )

        df_test_fe["target_scaled"] = scaler_y.transform(
            df_test_fe[["target"]]
        )

        X_train, y_train, _ = create_seq_data(
            df_train_fe,
            features,
            time_steps,
            "target_scaled"
        )

        X_test, y_test, test_seq_idx = create_seq_data(
            df_test_fe,
            features,
            time_steps,
            "target_scaled"
        )

        if (
            X_train.ndim != 3 or
            X_test.ndim != 3 or
            len(X_train) == 0 or
            len(X_test) == 0
        ):
            print(
                f"Skipping fold {fold}: not enough sequence data | "
                f"Target={target_var}, Horizon={horizon}, Feature={feature_set}, "
                f"Train rows={len(df_train_fe)}, Test rows={len(df_test_fe)}, "
                f"X_train={X_train.shape}, X_test={X_test.shape}"
            )
            continue

        try:
            X_fit, y_fit, X_val, y_val = make_train_val_split(
                X_train,
                y_train,
                val_fraction=0.15
            )
        except ValueError as e:
            print(f"Skipping fold {fold}: {e}")
            continue

        model = build_attention_rnn(
            input_shape=(X_train.shape[1], X_train.shape[2]),
            rnn_type=rnn_type
        )

        callbacks = [
            EarlyStopping(
                patience=5,
                restore_best_weights=True,
                monitor="val_loss"
            ),
            ReduceLROnPlateau(
                factor=0.5,
                patience=3,
                min_lr=1e-5,
                monitor="val_loss"
            )
        ]

        model.fit(
            X_fit,
            y_fit,
            validation_data=(X_val, y_val),
            epochs=100,
            batch_size=32,
            callbacks=callbacks,
            verbose=0
        )

        y_pred_scaled = model.predict(X_test, verbose=0).flatten()

        y_pred_change = scaler_y.inverse_transform(
            y_pred_scaled.reshape(-1, 1)
        ).flatten()

        y_test_change = scaler_y.inverse_transform(
            y_test.reshape(-1, 1)
        ).flatten()

        current_values = df_test_fe.loc[
            test_seq_idx,
            target_var
        ].reset_index(drop=True)

        y_pred_final = current_values + y_pred_change
        y_true_final = current_values + y_test_change

        rmse_list.append(
            np.sqrt(mean_squared_error(y_true_final, y_pred_final))
        )

        mae_list.append(
            mean_absolute_error(y_true_final, y_pred_final)
        )

        r2_list.append(
            r2_score(y_true_final, y_pred_final)
        )

        final_preds = {
            "index": pd.Index(test_seq_idx),
            "y_true": np.asarray(y_true_final),
            "y_pred": np.asarray(y_pred_final)
        }

        tf.keras.backend.clear_session()
        gc.collect()

    if len(rmse_list) == 0:
        return [np.nan], [np.nan], [np.nan], None

    return rmse_list, mae_list, r2_list, final_preds

In [ ]:
# ============================================================
# Run CV for LSTM+Attn and GRU+Attn
# ============================================================

set_seed(42)

TIME_STEPS = 15
N_SPLITS = 5

horizons = [1, 3, 7]
targets = ["Avg_Elapsed_Time", "Avg_Remaining_Time"]

rnn_cv_results = []
rnn_cv_predictions = {}

start_time = time.time()

for horizon in tqdm(horizons, desc="CV Horizons"):
    for target_var in tqdm(targets, desc=f"Targets H={horizon}", leave=False):

        for model_name, rnn_type in [
            ("LSTM+Attn", "lstm"),
            ("GRU+Attn", "gru")
        ]:

            rmse_b, mae_b, r2_b, preds_b = train_rnn_cv_for_target_horizon(
                df_train_full,
                target_var,
                horizon,
                feature_set="baseline",
                rnn_type=rnn_type,
                time_steps=TIME_STEPS,
                n_splits=N_SPLITS,
                seed=42
            )

            rmse_a, mae_a, r2_a, preds_a = train_rnn_cv_for_target_horizon(
                df_train_full,
                target_var,
                horizon,
                feature_set="actor",
                rnn_type=rnn_type,
                time_steps=TIME_STEPS,
                n_splits=N_SPLITS,
                seed=42
            )

            rnn_cv_results.append(
                summarize_model_comparison(
                    model_name,
                    target_var,
                    horizon,
                    rmse_b,
                    rmse_a,
                    mae_b,
                    mae_a,
                    r2_b,
                    r2_a
                )
            )

            rnn_cv_predictions[
                (model_name, target_var, horizon, "baseline")
            ] = preds_b

            rnn_cv_predictions[
                (model_name, target_var, horizon, "actor")
            ] = preds_a


rnn_cv_df = pd.DataFrame(rnn_cv_results).sort_values(
    ["Model", "Target", "Horizon"]
).reset_index(drop=True)

print("\nRNN CV Summary:")
print(rnn_cv_df.to_markdown(index=False))

print(f"\nFinished CV training in {(time.time() - start_time) / 60:.2f} minutes")


# ============================================================
# Save RNN CV Results
# ============================================================

SAVE_DIR_CV = "rnn_attention_cv_artifacts_2012"
os.makedirs(SAVE_DIR_CV, exist_ok=True)

rnn_cv_df.to_csv(
    f"{SAVE_DIR_CV}/rnn_attention_cv_results_2012.csv",
    index=False
)

joblib.dump(
    rnn_cv_predictions,
    f"{SAVE_DIR_CV}/rnn_attention_cv_predictions_2012.joblib"
)

joblib.dump(
    rnn_cv_results,
    f"{SAVE_DIR_CV}/rnn_attention_cv_raw_results_2012.joblib"
)

!zip -r rnn_attention_cv_artifacts_2012.zip rnn_attention_cv_artifacts_2012

from google.colab import files
files.download("rnn_attention_cv_artifacts_2012.zip")

In [ ]:
# ============================================================
# RNN WITH ATTENTION: Multi-Seed Holdout Evaluation
# ============================================================

def final_evaluate_rnn_for_target_horizon(
    df_train_full,
    df_test_final,
    target_var,
    horizon,
    feature_set="baseline",
    rnn_type="gru",
    time_steps=15,
    seed=42
):
    set_seed(seed)

    CONTEXT_ROWS = 200

    df_train_fe, baseline_features, actor_features = feature_engineering(
        df_train_full,
        target_var=target_var,
        horizon=horizon
    )

    df_context = pd.concat([
        df_train_full.tail(CONTEXT_ROWS),
        df_test_final
    ])

    df_test_fe, _, _ = feature_engineering(
        df_context,
        target_var=target_var,
        horizon=horizon
    )

    df_test_fe = df_test_fe.loc[
        df_test_final.index.intersection(df_test_fe.index)
    ]

    df_train_fe.dropna(inplace=True)
    df_test_fe.dropna(inplace=True)

    df_train_fe = df_train_fe.copy()
    df_test_fe = df_test_fe.copy()

    if feature_set == "baseline":
        features = baseline_features
    else:
        features = baseline_features + actor_features

    scaler_x = StandardScaler()
    scaler_y = StandardScaler()

    df_train_fe[features] = scaler_x.fit_transform(df_train_fe[features])
    df_test_fe[features] = scaler_x.transform(df_test_fe[features])

    df_train_fe["target_scaled"] = scaler_y.fit_transform(
        df_train_fe[["target"]]
    )

    df_test_fe["target_scaled"] = scaler_y.transform(
        df_test_fe[["target"]]
    )

    X_train, y_train, _ = create_seq_data(
        df_train_fe,
        features,
        time_steps,
        "target_scaled"
    )

    X_test, y_test, test_seq_idx = create_seq_data(
        df_test_fe,
        features,
        time_steps,
        "target_scaled"
    )

    if (
        X_train.ndim != 3 or
        X_test.ndim != 3 or
        len(X_train) == 0 or
        len(X_test) == 0
    ):
        raise ValueError(
            f"No valid holdout sequences created. "
            f"Target={target_var}, Horizon={horizon}, Feature={feature_set}, "
            f"Train rows={len(df_train_fe)}, Test rows={len(df_test_fe)}, "
            f"X_train={X_train.shape}, X_test={X_test.shape}"
        )

    X_fit, y_fit, X_val, y_val = make_train_val_split(
        X_train,
        y_train,
        val_fraction=0.15
    )

    model = build_attention_rnn(
        input_shape=(X_train.shape[1], X_train.shape[2]),
        rnn_type=rnn_type
    )

    callbacks = [
        EarlyStopping(
            patience=5,
            restore_best_weights=True,
            monitor="val_loss"
        ),
        ReduceLROnPlateau(
            factor=0.5,
            patience=3,
            min_lr=1e-5,
            monitor="val_loss"
        )
    ]

    model.fit(
        X_fit,
        y_fit,
        validation_data=(X_val, y_val),
        epochs=100,
        batch_size=32,
        callbacks=callbacks,
        verbose=0
    )

    y_pred_scaled = model.predict(X_test, verbose=0).flatten()

    y_pred_change = scaler_y.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    ).flatten()

    y_test_change = scaler_y.inverse_transform(
        y_test.reshape(-1, 1)
    ).flatten()

    current_values = df_test_fe.loc[
        test_seq_idx,
        target_var
    ].reset_index(drop=True)

    y_pred_final = current_values + y_pred_change
    y_true_final = current_values + y_test_change

    result = {
        "index": pd.Index(test_seq_idx),
        "y_true": np.asarray(y_true_final),
        "y_pred": np.asarray(y_pred_final)
    }

    tf.keras.backend.clear_session()
    gc.collect()

    return result

In [ ]:
# ============================================================
# Run Multi-Seed Holdout Evaluation
# ============================================================

seeds = [1, 7, 13, 21, 42, 66, 77, 88, 99, 123]

horizons = [1, 3, 7]
targets = ["Avg_Elapsed_Time", "Avg_Remaining_Time"]

multi_seed_rnn_results = []
rnn_holdout_predictions = {}

start_time = time.time()

for model_name, rnn_type in tqdm(
    [("LSTM+Attn", "lstm"), ("GRU+Attn", "gru")],
    desc="RNN Models"
):
    for horizon in tqdm(horizons, desc=f"{model_name} Horizons", leave=False):
        for target_var in tqdm(targets, desc=f"{model_name} H={horizon}", leave=False):

            for seed in tqdm(
                seeds,
                desc=f"{model_name} | {target_var} | H={horizon}",
                leave=False
            ):

                preds_b = final_evaluate_rnn_for_target_horizon(
                    df_train_full=df_train_full,
                    df_test_final=df_test_final,
                    target_var=target_var,
                    horizon=horizon,
                    feature_set="baseline",
                    rnn_type=rnn_type,
                    time_steps=TIME_STEPS,
                    seed=seed
                )

                preds_a = final_evaluate_rnn_for_target_horizon(
                    df_train_full=df_train_full,
                    df_test_final=df_test_final,
                    target_var=target_var,
                    horizon=horizon,
                    feature_set="actor",
                    rnn_type=rnn_type,
                    time_steps=TIME_STEPS,
                    seed=seed
                )

                y_true = np.asarray(preds_b["y_true"])
                pred_b = np.asarray(preds_b["y_pred"])
                pred_a = np.asarray(preds_a["y_pred"])

                rmse_b = np.sqrt(mean_squared_error(y_true, pred_b))
                rmse_a = np.sqrt(mean_squared_error(y_true, pred_a))

                mae_b = mean_absolute_error(y_true, pred_b)
                mae_a = mean_absolute_error(y_true, pred_a)

                r2_b = r2_score(y_true, pred_b)
                r2_a = r2_score(y_true, pred_a)

                multi_seed_rnn_results.append({
                    "Model": model_name,
                    "Target": target_var,
                    "Horizon": horizon,
                    "Seed": seed,

                    "RMSE Baseline": rmse_b,
                    "RMSE Actor": rmse_a,
                    "Delta RMSE": rmse_b - rmse_a,

                    "MAE Baseline": mae_b,
                    "MAE Actor": mae_a,
                    "Delta MAE": mae_b - mae_a,

                    "R2 Baseline": r2_b,
                    "R2 Actor": r2_a,
                    "Delta R2": r2_a - r2_b
                })

                rnn_holdout_predictions[
                    (model_name, target_var, horizon, seed, "baseline")
                ] = preds_b

                rnn_holdout_predictions[
                    (model_name, target_var, horizon, seed, "actor")
                ] = preds_a


multi_seed_rnn_df = pd.DataFrame(multi_seed_rnn_results)

print("\nRaw multi-seed RNN holdout results:")
print(multi_seed_rnn_df.to_markdown(index=False))

print(f"\nFinished multi-seed holdout evaluation in {(time.time() - start_time) / 60:.2f} minutes")

In [ ]:
# ============================================================
# RNN WITH ATTENTION: Seed-Level Wilcoxon Summary
# ============================================================

rnn_stat_rows = []

for model_name in ["LSTM+Attn", "GRU+Attn"]:
    for target_var in targets:
        for horizon in horizons:

            subset = multi_seed_rnn_df[
                (multi_seed_rnn_df["Model"] == model_name) &
                (multi_seed_rnn_df["Target"] == target_var) &
                (multi_seed_rnn_df["Horizon"] == horizon)
            ].copy()

            rmse_b = subset["RMSE Baseline"].values
            rmse_a = subset["RMSE Actor"].values

            mae_b = subset["MAE Baseline"].values
            mae_a = subset["MAE Actor"].values

            r2_b = subset["R2 Baseline"].values
            r2_a = subset["R2 Actor"].values

            try:
                _, p_rmse = wilcoxon(
                    rmse_b,
                    rmse_a,
                    alternative="greater"
                )
            except ValueError:
                p_rmse = np.nan

            try:
                _, p_mae = wilcoxon(
                    mae_b,
                    mae_a,
                    alternative="greater"
                )
            except ValueError:
                p_mae = np.nan

            rnn_stat_rows.append({
                "Model": model_name,
                "Target": target_var,
                "Horizon": horizon,

                "RMSE Baseline": fmt_mean_std(rmse_b),
                "RMSE Actor": fmt_mean_std(rmse_a),
                "Delta RMSE": f"{np.mean(rmse_b - rmse_a):.3f}",
                "RMSE p-value": f"{p_rmse:.4f}" if not np.isnan(p_rmse) else "NA",

                "MAE Baseline": fmt_mean_std(mae_b),
                "MAE Actor": fmt_mean_std(mae_a),
                "Delta MAE": f"{np.mean(mae_b - mae_a):.3f}",
                "MAE p-value": f"{p_mae:.4f}" if not np.isnan(p_mae) else "NA",

                "R2 Baseline": fmt_mean_std(r2_b),
                "R2 Actor": fmt_mean_std(r2_a),
                "Delta R2": f"{np.mean(r2_a - r2_b):.3f}"
            })


rnn_stats_df = pd.DataFrame(rnn_stat_rows)

print("\nRNN multi-seed holdout summary with seed-level Wilcoxon tests:")
print(rnn_stats_df.to_markdown(index=False))

In [ ]:
# ============================================================
# RNN WITH ATTENTION: Per-Observation Block Bootstrap Tests
# ============================================================

rnn_bootstrap_rows = []

BLOCK_SIZE = 10
N_BOOT = 5000

for model_name in ["LSTM+Attn", "GRU+Attn"]:
    for target_var in targets:
        for horizon in horizons:
            for seed in seeds:

                key_b = (model_name, target_var, horizon, seed, "baseline")
                key_a = (model_name, target_var, horizon, seed, "actor")

                if key_b not in rnn_holdout_predictions:
                    continue

                if key_a not in rnn_holdout_predictions:
                    continue

                preds_b = rnn_holdout_predictions[key_b]
                preds_a = rnn_holdout_predictions[key_a]

                test_result = paired_error_bootstrap_tests(
                    preds_b,
                    preds_a,
                    block_size=BLOCK_SIZE,
                    n_boot=N_BOOT,
                    seed=seed
                )

                rnn_bootstrap_rows.append({
                    "Model": model_name,
                    "Target": target_var,
                    "Horizon": horizon,
                    "Seed": seed,
                    "N Observations": test_result["N Observations"],

                    "Mean Delta Squared Error": test_result["Mean Delta Squared Error"],
                    "Squared Error Bootstrap p-value": test_result["Squared Error Bootstrap p-value"],

                    "Mean Delta Absolute Error": test_result["Mean Delta Absolute Error"],
                    "Absolute Error Bootstrap p-value": test_result["Absolute Error Bootstrap p-value"],
                })


rnn_bootstrap_df = pd.DataFrame(rnn_bootstrap_rows)

print("\nPer-seed paired error block bootstrap tests:")
print(rnn_bootstrap_df.to_markdown(index=False))


# ============================================================
# RNN WITH ATTENTION: Bootstrap Summary Across Seeds
# ============================================================

rnn_bootstrap_summary_rows = []

for model_name in ["LSTM+Attn", "GRU+Attn"]:
    for target_var in targets:
        for horizon in horizons:

            subset = rnn_bootstrap_df[
                (rnn_bootstrap_df["Model"] == model_name) &
                (rnn_bootstrap_df["Target"] == target_var) &
                (rnn_bootstrap_df["Horizon"] == horizon)
            ].copy()

            if len(subset) == 0:
                continue

            rnn_bootstrap_summary_rows.append({
                "Model": model_name,
                "Target": target_var,
                "Horizon": horizon,

                "Mean Delta Squared Error": fmt_mean_std(
                    subset["Mean Delta Squared Error"].values
                ),

                "Median Squared Error Bootstrap p-value": np.median(
                    subset["Squared Error Bootstrap p-value"].values
                ),

                "Mean Delta Absolute Error": fmt_mean_std(
                    subset["Mean Delta Absolute Error"].values
                ),

                "Median Absolute Error Bootstrap p-value": np.median(
                    subset["Absolute Error Bootstrap p-value"].values
                ),

                "Mean N Observations": int(
                    np.mean(subset["N Observations"].values)
                )
            })


rnn_bootstrap_summary_df = pd.DataFrame(rnn_bootstrap_summary_rows)

print("\nBootstrap summary across seeds:")
print(rnn_bootstrap_summary_df.to_markdown(index=False))

In [ ]:
# ============================================================
# Save RNN Multi-Seed Holdout + Bootstrap Results
# ============================================================

SAVE_DIR_HOLDOUT = "rnn_attention_multi_seed_holdout_2012"
os.makedirs(SAVE_DIR_HOLDOUT, exist_ok=True)

multi_seed_rnn_df.to_csv(
    f"{SAVE_DIR_HOLDOUT}/rnn_attention_multi_seed_raw_results_2012.csv",
    index=False
)

rnn_stats_df.to_csv(
    f"{SAVE_DIR_HOLDOUT}/rnn_attention_multi_seed_summary_stats_2012.csv",
    index=False
)

rnn_bootstrap_df.to_csv(
    f"{SAVE_DIR_HOLDOUT}/rnn_attention_per_seed_block_bootstrap_tests_2012.csv",
    index=False
)

rnn_bootstrap_summary_df.to_csv(
    f"{SAVE_DIR_HOLDOUT}/rnn_attention_block_bootstrap_summary_2012.csv",
    index=False
)

joblib.dump(
    rnn_holdout_predictions,
    f"{SAVE_DIR_HOLDOUT}/rnn_attention_multi_seed_predictions_models_2012.joblib"
)

!zip -r rnn_attention_multi_seed_holdout_2012.zip rnn_attention_multi_seed_holdout_2012

from google.colab import files
files.download("rnn_attention_multi_seed_holdout_2012.zip")

In [ ]:
import zipfile
import os

ZIP_PATH = "/content/rnn_attention_cv_artifacts_2012.zip"
EXTRACT_DIR = "/content/rnn_attention_cv_artifacts_2012"

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Extracted to:", EXTRACT_DIR)
print(os.listdir(EXTRACT_DIR))

In [ ]:
# ============================================================
# Mean SHAP over seeds for RNN Attention models
# Run AFTER holdout/statistical tests
# ============================================================

# Optional install:
# !pip install shap

import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

try:
    import shap
except ImportError:
    raise ImportError("Please run: !pip install shap")

SAVE_DIR = "rnn_mean_shap_outputs_2012"
os.makedirs(SAVE_DIR, exist_ok=True)

SHAP_SEEDS = [1, 7, 13, 21, 42]
SHAP_HORIZONS = [1, 3, 7]
SHAP_TARGETS = ["Avg_Elapsed_Time", "Avg_Remaining_Time"]
SHAP_FEATURE_SETS = ["baseline", "actor"]
SHAP_MODELS = [("LSTM+Attn", "lstm"), ("GRU+Attn", "gru")]

TIME_STEPS = 15
CONTEXT_ROWS = 200

N_BACKGROUND = 30
N_EXPLAIN = 10
TOP_N_FEATURES = 20


def train_rnn_for_shap(
    df_train_full,
    df_test_final,
    target_var,
    horizon,
    feature_set="actor",
    rnn_type="gru",
    time_steps=15,
    seed=42
):
    set_seed(seed)

    df_train_fe, baseline_features, actor_features = feature_engineering(
        df_train_full,
        target_var=target_var,
        horizon=horizon
    )

    df_context = pd.concat([df_train_full.tail(CONTEXT_ROWS), df_test_final])

    df_test_fe, _, _ = feature_engineering(
        df_context,
        target_var=target_var,
        horizon=horizon
    )

    df_test_fe = df_test_fe.loc[df_test_final.index.intersection(df_test_fe.index)]

    df_train_fe.dropna(inplace=True)
    df_test_fe.dropna(inplace=True)

    df_train_fe = df_train_fe.copy()
    df_test_fe = df_test_fe.copy()

    features = baseline_features if feature_set == "baseline" else baseline_features + actor_features

    scaler_x = StandardScaler()
    scaler_y = StandardScaler()

    df_train_fe[features] = scaler_x.fit_transform(df_train_fe[features])
    df_test_fe[features] = scaler_x.transform(df_test_fe[features])

    df_train_fe["target_scaled"] = scaler_y.fit_transform(df_train_fe[["target"]])
    df_test_fe["target_scaled"] = scaler_y.transform(df_test_fe[["target"]])

    X_train, y_train, _ = create_seq_data(
        df_train_fe,
        features,
        time_steps,
        "target_scaled"
    )

    X_test, y_test, test_seq_idx = create_seq_data(
        df_test_fe,
        features,
        time_steps,
        "target_scaled"
    )

    if X_train.ndim != 3 or X_test.ndim != 3 or len(X_train) == 0 or len(X_test) == 0:
        raise ValueError(
            f"No valid SHAP sequences. "
            f"Target={target_var}, Horizon={horizon}, Feature={feature_set}, "
            f"Seed={seed}, X_train={X_train.shape}, X_test={X_test.shape}"
        )

    model = build_attention_rnn(
        input_shape=(X_train.shape[1], X_train.shape[2]),
        rnn_type=rnn_type
    )

    callbacks = [
        EarlyStopping(patience=5, restore_best_weights=True),
        ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-5)
    ]

    model.fit(
        X_train,
        y_train,
        validation_data=(X_test, y_test),
        epochs=100,
        batch_size=32,
        callbacks=callbacks,
        verbose=0
    )

    return model, X_train, X_test, features


def compute_seed_shap_map(model, X_train, X_test):
    background = X_train[:min(N_BACKGROUND, len(X_train))]
    X_explain = X_test[:min(N_EXPLAIN, len(X_test))]

    explainer = shap.GradientExplainer(model, background)
    shap_values = explainer.shap_values(X_explain)

    if isinstance(shap_values, list):
        shap_values = shap_values[0]

    shap_values = np.array(shap_values)

    if shap_values.ndim == 4:
        shap_values = shap_values[..., 0]

    # samples x time_steps x features -> time_steps x features
    mean_abs_shap = np.mean(np.abs(shap_values), axis=0)

    return mean_abs_shap


all_shap_summary_rows = []

for model_name, rnn_type in SHAP_MODELS:
    for horizon in SHAP_HORIZONS:
        for target_var in SHAP_TARGETS:
            for feature_set in SHAP_FEATURE_SETS:

                print(
                    f"\nRunning mean SHAP: "
                    f"{model_name} | {target_var} | H={horizon} | {feature_set}"
                )

                seed_maps = []
                features_ref = None

                for seed in SHAP_SEEDS:
                    print(f"  Seed {seed}")

                    try:
                        model, X_train, X_test, features = train_rnn_for_shap(
                            df_train_full=df_train_full,
                            df_test_final=df_test_final,
                            target_var=target_var,
                            horizon=horizon,
                            feature_set=feature_set,
                            rnn_type=rnn_type,
                            time_steps=TIME_STEPS,
                            seed=seed
                        )

                        mean_abs_shap = compute_seed_shap_map(
                            model,
                            X_train,
                            X_test
                        )

                        seed_maps.append(mean_abs_shap)
                        features_ref = features

                    except Exception as e:
                        print(f"  Skipping seed {seed} because SHAP failed:")
                        print(" ", e)

                    finally:
                        tf.keras.backend.clear_session()
                        gc.collect()

                if len(seed_maps) == 0:
                    print("  No successful SHAP runs for this configuration.")
                    continue

                mean_shap_over_seeds = np.mean(seed_maps, axis=0)

                feature_importance = np.mean(mean_shap_over_seeds, axis=0)
                time_importance = np.mean(mean_shap_over_seeds, axis=1)

                safe_name = (
                    f"{model_name}_{target_var}_H{horizon}_{feature_set}"
                    .replace("+", "")
                    .replace(" ", "_")
                    .replace("/", "_")
                )

                feature_importance_df = pd.DataFrame({
                    "Feature": features_ref,
                    "Mean_abs_SHAP_over_seeds": feature_importance
                }).sort_values("Mean_abs_SHAP_over_seeds", ascending=False)

                time_importance_df = pd.DataFrame({
                    "Relative_time_step": np.arange(-TIME_STEPS + 1, 1),
                    "Mean_abs_SHAP_over_seeds": time_importance
                })

                feature_importance_df.to_csv(
                    f"{SAVE_DIR}/{safe_name}_feature_importance.csv",
                    index=False
                )

                time_importance_df.to_csv(
                    f"{SAVE_DIR}/{safe_name}_time_importance.csv",
                    index=False
                )

                np.save(
                    f"{SAVE_DIR}/{safe_name}_feature_time_shap.npy",
                    mean_shap_over_seeds
                )

                for _, row in feature_importance_df.iterrows():
                    all_shap_summary_rows.append({
                        "Model": model_name,
                        "Target": target_var,
                        "Horizon": horizon,
                        "Feature Set": feature_set,
                        "Feature": row["Feature"],
                        "Mean Abs SHAP Over Seeds": row["Mean_abs_SHAP_over_seeds"]
                    })

                # ------------------------------------------------------------
                # Plot 1: top feature importance
                # ------------------------------------------------------------

                top_features = feature_importance_df.head(TOP_N_FEATURES).iloc[::-1]

                plt.figure(figsize=(10, 7))
                plt.barh(
                    top_features["Feature"],
                    top_features["Mean_abs_SHAP_over_seeds"]
                )
                plt.xlabel("Mean absolute SHAP over seeds")
                plt.ylabel("Feature")
                plt.title(
                    f"Mean SHAP Feature Importance over Seeds\n"
                    f"{model_name} | {target_var} | H={horizon} | {feature_set}"
                )
                plt.tight_layout()
                plt.savefig(
                    f"{SAVE_DIR}/{safe_name}_top_features.png",
                    dpi=300,
                    bbox_inches="tight"
                )
                plt.show()

                # ------------------------------------------------------------
                # Plot 2: time-step importance
                # ------------------------------------------------------------

                plt.figure(figsize=(9, 5))
                plt.plot(
                    time_importance_df["Relative_time_step"],
                    time_importance_df["Mean_abs_SHAP_over_seeds"],
                    marker="o"
                )
                plt.xlabel("Relative time step")
                plt.ylabel("Mean absolute SHAP over seeds")
                plt.title(
                    f"Mean SHAP Time-step Importance over Seeds\n"
                    f"{model_name} | {target_var} | H={horizon} | {feature_set}"
                )
                plt.grid(True)
                plt.tight_layout()
                plt.savefig(
                    f"{SAVE_DIR}/{safe_name}_time_importance.png",
                    dpi=300,
                    bbox_inches="tight"
                )
                plt.show()

                # ------------------------------------------------------------
                # Plot 3: feature × time heatmap
                # ------------------------------------------------------------

                top_feature_names = feature_importance_df.head(TOP_N_FEATURES)["Feature"].tolist()
                top_feature_indices = [features_ref.index(f) for f in top_feature_names]

                heatmap_values = mean_shap_over_seeds[:, top_feature_indices]

                plt.figure(figsize=(13, 8))
                plt.imshow(heatmap_values.T, aspect="auto")
                plt.colorbar(label="Mean absolute SHAP over seeds")

                plt.xticks(
                    ticks=np.arange(TIME_STEPS),
                    labels=np.arange(-TIME_STEPS + 1, 1)
                )

                plt.yticks(
                    ticks=np.arange(len(top_feature_names)),
                    labels=top_feature_names
                )

                plt.xlabel("Relative time step")
                plt.ylabel("Feature")
                plt.title(
                    f"Mean SHAP Feature × Time Heatmap over Seeds\n"
                    f"{model_name} | {target_var} | H={horizon} | {feature_set}"
                )
                plt.tight_layout()
                plt.savefig(
                    f"{SAVE_DIR}/{safe_name}_feature_time_heatmap.png",
                    dpi=300,
                    bbox_inches="tight"
                )
                plt.show()


rnn_mean_shap_summary_df = pd.DataFrame(all_shap_summary_rows)

rnn_mean_shap_summary_df.to_csv(
    f"{SAVE_DIR}/rnn_mean_shap_summary_all_models.csv",
    index=False
)

print("\nSaved mean SHAP outputs to:", SAVE_DIR)
print("\nTop rows:")
print(rnn_mean_shap_summary_df.head(20).to_markdown(index=False))

tuning

In [ ]:
import numpy as np
import pandas as pd
from itertools import product
import tensorflow as tf
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Bidirectional, GRU, LSTM, Dropout, MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D, Dense)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import ttest_rel, sem, t

# Assume this function exists and works correctly
#from feature_engineering import feature_engineering

import os
import random

def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()

set_seed(42)



def build_attention_model(input_shape, rnn_units=128, dense_units=64, dropout=0.3, rnn_type='gru',
                          conv_filters=64, kernel_size=3, pool_size=2):
    input_layer = Input(shape=input_shape)
    x = Conv1D(filters=conv_filters, kernel_size=kernel_size, activation='relu', padding='same')(input_layer)
    x = MaxPooling1D(pool_size=pool_size)(x)
    x = Bidirectional(GRU(rnn_units, return_sequences=True) if rnn_type == 'gru' else LSTM(rnn_units, return_sequences=True))(x)
    x = Dropout(dropout)(x)
    x = (GRU(dense_units, return_sequences=True) if rnn_type == 'gru' else LSTM(dense_units, return_sequences=True))(x)
    x = Dropout(dropout)(x)
    x = MultiHeadAttention(num_heads=4, key_dim=32)(x, x)
    x = LayerNormalization(epsilon=1e-6)(x)
    x = GlobalAveragePooling1D()(x)
    output = Dense(1)(x)
    model = Model(inputs=input_layer, outputs=output)
    model.compile(optimizer='adam', loss='mse')
    return model



def create_seq_data(df, features, time_steps=15, target_col='target'):
    X, y = [], []
    for i in range(time_steps, len(df) - 1):
        X.append(df[features].iloc[i - time_steps:i].values)
        y.append(df[target_col].iloc[i + 1])
    return np.array(X), np.array(y)


def train_attention_rnn(df, features, rnn_type='gru', time_steps=15, n_splits=5, custom_params=None):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    rmse_list, mae_list = [], []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
        print(f"\n🔁 Fold {fold + 1}")

        df_train_raw = df.iloc[:test_idx[0]].copy()
        df_test_raw = df.iloc[test_idx].copy()

        df_train_fe, _, _ = feature_engineering(df_train_raw, residual_target=True)
        df_context = pd.concat([df_train_raw.tail(40), df_test_raw])
        df_test_fe, _, _ = feature_engineering(df_context, residual_target=True)
        df_test_fe = df_test_fe.loc[df_test_raw.index.intersection(df_test_fe.index)]

        df_train_fe.dropna(inplace=True)
        df_test_fe.dropna(inplace=True)

        scaler_x = StandardScaler()
        scaler_y = StandardScaler()
        df_train_fe[features] = scaler_x.fit_transform(df_train_fe[features])
        df_test_fe[features] = scaler_x.transform(df_test_fe[features])
        df_train_fe['target_scaled'] = scaler_y.fit_transform(df_train_fe[['target']])
        df_test_fe['target_scaled'] = scaler_y.transform(df_test_fe[['target']])

        X_train, y_train = create_seq_data(df_train_fe, features, time_steps, 'target_scaled')
        X_test, y_test = create_seq_data(df_test_fe, features, time_steps, 'target_scaled')

        base_values = df_test_fe['TT'].iloc[time_steps + 1: time_steps + 1 + len(y_test)].reset_index(drop=True)

        model = build_attention_model(
            input_shape=(X_train.shape[1], X_train.shape[2]),
            rnn_units=custom_params.get('rnn_units', 128),
            dense_units=custom_params.get('dense_units', 64),
            dropout=custom_params.get('dropout', 0.3),
            rnn_type=rnn_type,
            conv_filters=custom_params.get('conv_filters', 64),
            kernel_size=custom_params.get('kernel_size', 3),
            pool_size=custom_params.get('pool_size', 2)
         )


        callbacks = [
            EarlyStopping(patience=5, restore_best_weights=True),
            ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-5)
        ]

        model.fit(X_train, y_train, validation_data=(X_test, y_test),
                  epochs=100, batch_size=custom_params.get('batch_size', 16),
                  callbacks=callbacks, verbose=0)

        y_pred_scaled = model.predict(X_test).flatten()
        y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
        y_test_inv = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

        y_pred_final = base_values + y_pred
        y_true_final = base_values + y_test_inv

        rmse = np.sqrt(mean_squared_error(y_true_final, y_pred_final))
        mae = mean_absolute_error(y_true_final, y_pred_final)
        rmse_list.append(rmse)
        mae_list.append(mae)

    return rmse_list, mae_list


# Load data
print("\n📦 Loading data and setting up parameter grid")
df = df_train_full.copy()
df_model, baseline_features, actor_features = feature_engineering(df, residual_target=True)

param_grid = {
    'rnn_units': [64, 128],
    'dense_units': [32, 64],
    'dropout': [0.2, 0.3],
    'batch_size': [16, 32],
    'conv_filters': [32, 64],
    'kernel_size': [3, 5],
    'pool_size': [2, 3]
}

param_combos = list(product(*param_grid.values()))
param_names = list(param_grid.keys())

results_summary = []

for i, combo in enumerate(param_combos):
    set_seed(42)
    params = dict(zip(param_names, combo))
    print(f"\n\n🔧 Config {i+1}/{len(param_combos)}: {params}")

    rmse_b, mae_b = train_attention_rnn(df, baseline_features, rnn_type='gru', custom_params=params)
    rmse_a, mae_a = train_attention_rnn(df, baseline_features + actor_features, rnn_type='gru', custom_params=params)

    delta = np.mean(rmse_b) - np.mean(rmse_a)
    results_summary.append({
        'params': params,
        'rmse_baseline': np.mean(rmse_b),
        'rmse_actor': np.mean(rmse_a),
        'delta': delta,
        'outperforms': delta > 0
    })

    print(f"✅ Δ RMSE = {delta:.4f} | {'Actor Wins ✅' if delta > 0 else 'Baseline Wins ❌'}")

# Summary
summary_df = pd.DataFrame(results_summary).sort_values(by='delta', ascending=False)
print("\n\n📋 Final Results Summary:")
print(summary_df[['params', 'rmse_baseline', 'rmse_actor', 'delta', 'outperforms']].to_markdown(index=False))

print("\n✅ Configurations where Actor-Enriched outperformed Baseline:")
print(summary_df[summary_df['outperforms']][['params', 'rmse_baseline', 'rmse_actor', 'delta']].to_markdown(index=False))


In [ ]:
print("\n✅ Configurations where Actor-Enriched outperformed Baseline:")
print(summary_df[summary_df['outperforms']][['params', 'rmse_baseline', 'rmse_actor', 'delta']].to_markdown(index=False))
#choose mostly appearing configurations -> rnn_units 64 , dense 32, dropout 0.2 batch size 32

In [ ]:
from itertools import product
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, Bidirectional,
    GRU, LSTM, Dropout, MultiHeadAttention, LayerNormalization,
    GlobalAveragePooling1D, Dense
)

import os
import random

def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()

set_seed(42)

def build_attention_model(input_shape, rnn_units=128, dense_units=64, dropout=0.3, rnn_type='gru',
                          conv_filters=64, kernel_size=3, pool_size=2):
    input_layer = Input(shape=input_shape)
    x = Conv1D(filters=conv_filters, kernel_size=kernel_size, activation='relu', padding='same')(input_layer)
    x = MaxPooling1D(pool_size=pool_size)(x)
    x = Bidirectional(GRU(rnn_units, return_sequences=True) if rnn_type == 'gru' else LSTM(rnn_units, return_sequences=True))(x)
    x = Dropout(dropout)(x)
    x = (GRU(dense_units, return_sequences=True) if rnn_type == 'gru' else LSTM(dense_units, return_sequences=True))(x)
    x = Dropout(dropout)(x)
    x = MultiHeadAttention(num_heads=4, key_dim=32)(x, x)
    x = LayerNormalization(epsilon=1e-6)(x)
    x = GlobalAveragePooling1D()(x)
    output = Dense(1)(x)
    model = Model(inputs=input_layer, outputs=output)
    model.compile(optimizer='adam', loss='mse')
    return model

# === Sequence Creator ===
def create_seq_data(df, features, time_steps=15, target_col='target'):
    X, y = [], []
    for i in range(time_steps, len(df) - 1):
        X.append(df[features].iloc[i - time_steps:i].values)
        y.append(df[target_col].iloc[i + 1])
    return np.array(X), np.array(y)

# === Training Function ===
def train_attention_rnn(df, features, rnn_type='lstm', time_steps=15, n_splits=5, custom_params=None):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    rmse_list, mae_list = [], []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
        print(f"\n🔁 Fold {fold + 1}")

        df_train_raw = df.iloc[:test_idx[0]].copy()
        df_test_raw = df.iloc[test_idx].copy()

        df_train_fe, _, _ = feature_engineering(df_train_raw, residual_target=True)
        df_context = pd.concat([df_train_raw.tail(40), df_test_raw])
        df_test_fe, _, _ = feature_engineering(df_context, residual_target=True)
        df_test_fe = df_test_fe.loc[df_test_raw.index.intersection(df_test_fe.index)]

        df_train_fe.dropna(inplace=True)
        df_test_fe.dropna(inplace=True)

        scaler_x = StandardScaler()
        scaler_y = StandardScaler()
        df_train_fe[features] = scaler_x.fit_transform(df_train_fe[features])
        df_test_fe[features] = scaler_x.transform(df_test_fe[features])
        df_train_fe['target_scaled'] = scaler_y.fit_transform(df_train_fe[['target']])
        df_test_fe['target_scaled'] = scaler_y.transform(df_test_fe[['target']])

        X_train, y_train = create_seq_data(df_train_fe, features, time_steps, 'target_scaled')
        X_test, y_test = create_seq_data(df_test_fe, features, time_steps, 'target_scaled')
        base_values = df_test_fe['TT'].iloc[time_steps + 1: time_steps + 1 + len(y_test)].reset_index(drop=True)

        model = build_attention_model(
            input_shape=(X_train.shape[1], X_train.shape[2]),
            rnn_units=custom_params.get('rnn_units', 128),
            dense_units=custom_params.get('dense_units', 64),
            dropout=custom_params.get('dropout', 0.3),
            rnn_type=rnn_type,
            conv_filters=custom_params.get('conv_filters', 64),
            kernel_size=custom_params.get('kernel_size', 3),
            pool_size=custom_params.get('pool_size', 2)
         )

        callbacks = [
            EarlyStopping(patience=5, restore_best_weights=True),
            ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-5)
        ]

        model.fit(X_train, y_train, validation_data=(X_test, y_test),
                  epochs=100, batch_size=custom_params.get('batch_size', 16),
                  callbacks=callbacks, verbose=0)

        y_pred_scaled = model.predict(X_test).flatten()
        y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
        y_test_inv = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

        y_pred_final = base_values + y_pred
        y_true_final = base_values + y_test_inv

        rmse = np.sqrt(mean_squared_error(y_true_final, y_pred_final))
        mae = mean_absolute_error(y_true_final, y_pred_final)
        rmse_list.append(rmse)
        mae_list.append(mae)

    return rmse_list, mae_list

# === Run LSTM Grid Search ===
print("\n📦 Running LSTM hyperparameter tuning")

df = df_train_full.copy()
df_model, baseline_features, actor_features = feature_engineering(df, residual_target=True)

param_grid = {
    'rnn_units': [64, 128],
    'dense_units': [32, 64],
    'dropout': [0.2, 0.3],
    'batch_size': [16, 32],
    'conv_filters': [32, 64],
    'kernel_size': [3, 5],
    'pool_size': [2, 3]
}
param_combos = list(product(*param_grid.values()))
param_names = list(param_grid.keys())

results_summary = []

for i, combo in enumerate(param_combos):
    set_seed(42)
    params = dict(zip(param_names, combo))
    print(f"\n\n🔧 Config {i+1}/{len(param_combos)} | Params: {params}")

    rmse_b, mae_b = train_attention_rnn(df, baseline_features, rnn_type='lstm', custom_params=params)
    rmse_a, mae_a = train_attention_rnn(df, baseline_features + actor_features, rnn_type='lstm', custom_params=params)

    delta = np.mean(rmse_b) - np.mean(rmse_a)
    results_summary.append({
        'params': params,
        'rmse_baseline': np.mean(rmse_b),
        'rmse_actor': np.mean(rmse_a),
        'delta': delta,
        'outperforms': delta > 0
    })

    print(f"✅ Δ RMSE = {delta:.4f} | {'Actor Wins ✅' if delta > 0 else 'Baseline Wins ❌'}")

# === Print Summary ===
summary_df = pd.DataFrame(results_summary).sort_values(by='delta', ascending=False)
print("\n\n📋 LSTM Tuning Summary:")
print(summary_df[['params', 'rmse_baseline', 'rmse_actor', 'delta', 'outperforms']].to_markdown(index=False))

print("\n✅ Actor-Enriched configurations that outperformed Baseline:")
print(summary_df[summary_df['outperforms']][['params', 'rmse_baseline', 'rmse_actor', 'delta']].to_markdown(index=False))


In [ ]:
print("\n✅ Actor-Enriched configurations that outperformed Baseline:")
print(summary_df[summary_df['outperforms']][['params', 'rmse_baseline', 'rmse_actor', 'delta']].to_markdown(index=False))
#choose mostly appearing configurations -> rnn_units 64 , dense 32, dropout 0.2 batch size 32

# RNN without attention

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import (
    LSTM, GRU, Dense, Dropout, Input, Bidirectional,
    Conv1D, MaxPooling1D, GlobalAveragePooling1D,
    LayerNormalization, MultiHeadAttention
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel, sem, t

import os
import random

In [ ]:
#rnn_without_attention_model.py
def build_no_attention_rnn(input_shape, rnn_type="gru"):
    input_layer = Input(shape=input_shape)

    x = Conv1D(filters=64, kernel_size=3, activation="relu", padding="same")(input_layer)
    x = MaxPooling1D(pool_size=2)(x)

    if rnn_type == "lstm":
        x = Bidirectional(LSTM(64, return_sequences=True))(x)
        x = Dropout(0.2)(x)
        x = LSTM(32, return_sequences=True)(x)
    else:
        x = Bidirectional(GRU(64, return_sequences=True))(x)
        x = Dropout(0.2)(x)
        x = GRU(32, return_sequences=True)(x)

    x = Dropout(0.2)(x)
    x = GlobalAveragePooling1D()(x)
    output = Dense(1)(x)

    model = Model(inputs=input_layer, outputs=output)
    model.compile(optimizer="adam", loss="mse")
    return model

In [ ]:
# -----------------------------
# Helper Functions
# -----------------------------

def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()

set_seed(42)

def create_seq_data(df, features, time_steps=15, target_col="target_scaled"):
    X, y, idx = [], [], []
    for i in range(time_steps, len(df)):
        X.append(df[features].iloc[i - time_steps:i].values)
        y.append(df[target_col].iloc[i])
        idx.append(df.index[i])
    return np.array(X), np.array(y), idx

def cohen_d(x, y):
    diff = np.array(x) - np.array(y)
    return diff.mean() / diff.std(ddof=1)

def confidence_interval(data, confidence=0.95):
    m = np.mean(data)
    se = sem(data)
    h = se * t.ppf((1 + confidence) / 2., len(data)-1)
    return f"{m:.3f} ± {h:.3f}"

def summarize_model_comparison(name, target_name, horizon, rmse_b, rmse_a, mae_b, mae_a, r2_b, r2_a):
    stat, p_rmse = ttest_rel(rmse_b, rmse_a)

    return {
        "Model": name,
        "Target": target_name,
        "Horizon": horizon,
        "RMSE Baseline": confidence_interval(rmse_b),
        "RMSE Actor": confidence_interval(rmse_a),
        "RMSE Δ": f"{np.mean(rmse_b) - np.mean(rmse_a):.3f}",
        "MAE Baseline": confidence_interval(mae_b),
        "MAE Actor": confidence_interval(mae_a),
        "MAE Δ": f"{np.mean(mae_b) - np.mean(mae_a):.3f}",
        "R² Baseline": confidence_interval(r2_b),
        "R² Actor": confidence_interval(r2_a),
        "R² Δ": f"{np.mean(r2_a) - np.mean(r2_b):.3f}",
        "p-value (RMSE)": f"{p_rmse:.4f}",
        "Cohen’s d": f"{cohen_d(rmse_b, rmse_a):.3f}",
    }

def bootstrap_metrics(y_true, y_pred, n_bootstrap=1000, seed=42):
    rng = np.random.RandomState(seed)
    rmse_vals, mae_vals, r2_vals = [], [], []
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n = len(y_true)

    for _ in range(n_bootstrap):
        idx = rng.choice(n, n, replace=True)
        yt_bs = y_true[idx]
        yp_bs = y_pred[idx]
        rmse_vals.append(np.sqrt(mean_squared_error(yt_bs, yp_bs)))
        mae_vals.append(mean_absolute_error(yt_bs, yp_bs))
        r2_vals.append(r2_score(yt_bs, yp_bs))

    return {
        "rmse": (np.mean(rmse_vals), np.std(rmse_vals)),
        "mae": (np.mean(mae_vals), np.std(mae_vals)),
        "r2": (np.mean(r2_vals), np.std(r2_vals)),
    }

def fmt(mean, std):
 return f"{mean:.3f} ± {std:.3f}"

def plot_rnn_cv_predictions(rnn_cv_predictions, model_name, target_var, horizon):
    preds_b = rnn_cv_predictions[(model_name, target_var, horizon, "baseline")]
    preds_a = rnn_cv_predictions[(model_name, target_var, horizon, "actor")]

    plt.figure(figsize=(14, 6))
    plt.plot(preds_b["index"], preds_b["y_true"], label="Actual", color="black")
    plt.plot(preds_b["index"], preds_b["y_pred"], "--", label=f"{model_name} Baseline", color="blue")
    plt.plot(preds_a["index"], preds_a["y_pred"], "--", label=f"{model_name} Actor-Enriched", color="green")
    plt.title(f"{model_name} CV Predictions | Target={target_var} | Horizon={horizon}")
    plt.xlabel("Time")
    plt.ylabel(target_var)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
#RNN without attention.py
#from feature_engineering import feature_engineering

def train_rnn_cv_for_target_horizon(
    df_train_full,
    target_var,
    horizon,
    feature_set="baseline",   # "baseline" or "actor"
    rnn_type="gru",
    time_steps=15,
    n_splits=5
):
    tscv = TimeSeriesSplit(n_splits=n_splits)

    rmse_list, mae_list, r2_list = [], [], []
    final_preds = None

    for fold, (train_idx, test_idx) in enumerate(tscv.split(df_train_full)):
        print(f"🔁 Fold {fold + 1}")

        df_train_raw = df_train_full.iloc[train_idx].copy()
        df_test_raw = df_train_full.iloc[test_idx].copy()

        df_train_fe, baseline_features, actor_features = feature_engineering(
            df_train_raw,
            target_var=target_var,
            horizon=horizon
        )

        df_context = pd.concat([df_train_raw.tail(50), df_test_raw])
        df_test_fe, _, _ = feature_engineering(
            df_context,
            target_var=target_var,
            horizon=horizon
        )
        df_test_fe = df_test_fe.loc[df_test_raw.index.intersection(df_test_fe.index)]

        df_train_fe.dropna(inplace=True)
        df_test_fe.dropna(inplace=True)

        min_required_rows = time_steps + 1

        if len(df_train_fe) < min_required_rows or len(df_test_fe) < min_required_rows:
            print(
                f"⚠️ Skipping fold {fold + 1}: not enough rows after feature engineering "
                f"(train={len(df_train_fe)}, test={len(df_test_fe)}, need at least {min_required_rows})"
            )
            continue

        features = baseline_features if feature_set == "baseline" else baseline_features + actor_features

        scaler_x = StandardScaler()
        scaler_y = StandardScaler()

        df_train_fe[features] = scaler_x.fit_transform(df_train_fe[features])
        df_test_fe[features] = scaler_x.transform(df_test_fe[features])

        df_train_fe["target_scaled"] = scaler_y.fit_transform(df_train_fe[["target"]])
        df_test_fe["target_scaled"] = scaler_y.transform(df_test_fe[["target"]])

        X_train, y_train, train_seq_idx = create_seq_data(df_train_fe, features, time_steps, "target_scaled")
        X_test, y_test, test_seq_idx = create_seq_data(df_test_fe, features, time_steps, "target_scaled")

        model = build_no_attention_rnn((X_train.shape[1], X_train.shape[2]), rnn_type=rnn_type)

        callbacks = [
            EarlyStopping(patience=5, restore_best_weights=True),
            ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-5)
        ]

        model.fit(
            X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=100,
            batch_size=32,
            callbacks=callbacks,
            verbose=0
        )

        y_pred_scaled = model.predict(X_test, verbose=0).flatten()
        y_pred_change = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
        y_test_change = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

        current_values = df_test_fe.loc[test_seq_idx, target_var].reset_index(drop=True)

        y_pred_final = current_values + y_pred_change
        y_true_final = current_values + y_test_change

        rmse_list.append(np.sqrt(mean_squared_error(y_true_final, y_pred_final)))
        mae_list.append(mean_absolute_error(y_true_final, y_pred_final))
        r2_list.append(r2_score(y_true_final, y_pred_final))

        if fold == n_splits - 1:
            final_preds = {
                "index": pd.Index(test_seq_idx),
                "y_true": y_true_final,
                "y_pred": y_pred_final
            }

    return rmse_list, mae_list, r2_list, final_preds

def final_evaluate_rnn_for_target_horizon(
    df_train_full,
    df_test_final,
    target_var,
    horizon,
    feature_set="baseline",
    rnn_type="gru",
    time_steps=15
):
    df_train_fe, baseline_features, actor_features = feature_engineering(
        df_train_full,
        target_var=target_var,
        horizon=horizon
    )

    df_context = pd.concat([df_train_full.tail(50), df_test_final])
    df_test_fe, _, _ = feature_engineering(
        df_context,
        target_var=target_var,
        horizon=horizon
    )
    df_test_fe = df_test_fe.loc[df_test_final.index.intersection(df_test_fe.index)]

    df_train_fe.dropna(inplace=True)
    df_test_fe.dropna(inplace=True)

    features = baseline_features if feature_set == "baseline" else baseline_features + actor_features

    scaler_x = StandardScaler()
    scaler_y = StandardScaler()

    df_train_fe[features] = scaler_x.fit_transform(df_train_fe[features])
    df_test_fe[features] = scaler_x.transform(df_test_fe[features])

    df_train_fe["target_scaled"] = scaler_y.fit_transform(df_train_fe[["target"]])
    df_test_fe["target_scaled"] = scaler_y.transform(df_test_fe[["target"]])

    X_train, y_train, _ = create_seq_data(df_train_fe, features, time_steps, "target_scaled")
    X_test, y_test, test_seq_idx = create_seq_data(df_test_fe, features, time_steps, "target_scaled")

    model = build_no_attention_rnn((X_train.shape[1], X_train.shape[2]), rnn_type=rnn_type)

    callbacks = [
        EarlyStopping(patience=5, restore_best_weights=True),
        ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-5)
    ]

    model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=100,
        batch_size=32,
        callbacks=callbacks,
        verbose=0
    )

    y_pred_scaled = model.predict(X_test, verbose=0).flatten()
    y_pred_change = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    y_test_change = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

    current_values = df_test_fe.loc[test_seq_idx, target_var].reset_index(drop=True)

    y_pred_final = current_values + y_pred_change
    y_true_final = current_values + y_test_change

    metrics = bootstrap_metrics(y_true_final, y_pred_final, seed=42)

    return {
        "index": pd.Index(test_seq_idx),
        "y_true": y_true_final,
        "y_pred": y_pred_final,
        "metrics": metrics,
        "model": model,
        "features": features,
        "X_test": X_test
    }
# -------------------------
# 🚀 Train All RNN Models
# -------------------------
set_seed(42)

horizons = [1, 3, 7]
targets = ["Avg_Elapsed_Time", "Avg_Remaining_Time", "WIP"]

rnn_cv_results = []
rnn_cv_predictions = {}

for horizon in horizons:
    for target_var in targets:
        print(f"\n==============================")
        print(f"RNN CV | Target: {target_var} | Horizon: {horizon}")
        print(f"==============================")

        # LSTM
        rmse_b_lstm, mae_b_lstm, r2_b_lstm, preds_b_lstm = train_rnn_cv_for_target_horizon(
            df_train_full, target_var, horizon, feature_set="baseline", rnn_type="lstm"
        )
        rmse_a_lstm, mae_a_lstm, r2_a_lstm, preds_a_lstm = train_rnn_cv_for_target_horizon(
            df_train_full, target_var, horizon, feature_set="actor", rnn_type="lstm"
        )

        rnn_cv_results.append(
            summarize_model_comparison(
                "LSTM", target_var, horizon,
                rmse_b_lstm, rmse_a_lstm,
                mae_b_lstm, mae_a_lstm,
                r2_b_lstm, r2_a_lstm
            )
        )

        rnn_cv_predictions[("LSTM", target_var, horizon, "baseline")] = preds_b_lstm
        rnn_cv_predictions[("LSTM", target_var, horizon, "actor")] = preds_a_lstm

        # GRU
        rmse_b_gru, mae_b_gru, r2_b_gru, preds_b_gru = train_rnn_cv_for_target_horizon(
            df_train_full, target_var, horizon, feature_set="baseline", rnn_type="gru"
        )
        rmse_a_gru, mae_a_gru, r2_a_gru, preds_a_gru = train_rnn_cv_for_target_horizon(
            df_train_full, target_var, horizon, feature_set="actor", rnn_type="gru"
        )

        rnn_cv_results.append(
            summarize_model_comparison(
                "GRU", target_var, horizon,
                rmse_b_gru, rmse_a_gru,
                mae_b_gru, mae_a_gru,
                r2_b_gru, r2_a_gru
            )
        )

        rnn_cv_predictions[("GRU", target_var, horizon, "baseline")] = preds_b_gru
        rnn_cv_predictions[("GRU", target_var, horizon, "actor")] = preds_a_gru

rnn_cv_df = pd.DataFrame(rnn_cv_results).sort_values(["Model", "Target", "Horizon"]).reset_index(drop=True)
print(rnn_cv_df.to_markdown(index=False))

# ===============================
# 📈 Plot Actual vs Predictions
# ===============================

plot_rnn_cv_predictions(rnn_cv_predictions, "LSTM", "Avg_Elapsed_Time", 1)
plot_rnn_cv_predictions(rnn_cv_predictions, "GRU", "Avg_Elapsed_Time", 1)

In [ ]:
rnn_holdout_results = []
rnn_holdout_predictions = {}

for horizon in horizons:
    for target_var in targets:
        for rnn_type in ["lstm", "gru"]:
            print(f"\n==============================")
            print(f"RNN HOLDOUT | Model: {rnn_type.upper()} | Target: {target_var} | Horizon: {horizon}")
            print(f"==============================")

            preds_b = final_evaluate_rnn_for_target_horizon(
                df_train_full, df_test_final,
                target_var=target_var,
                horizon=horizon,
                feature_set="baseline",
                rnn_type=rnn_type
            )

            preds_a = final_evaluate_rnn_for_target_horizon(
                df_train_full, df_test_final,
                target_var=target_var,
                horizon=horizon,
                feature_set="actor",
                rnn_type=rnn_type
            )

            rb, rs = preds_b["metrics"]["rmse"]
            ra, ras = preds_a["metrics"]["rmse"]
            mb, ms = preds_b["metrics"]["mae"]
            ma, mas = preds_a["metrics"]["mae"]
            r2b, r2bs = preds_b["metrics"]["r2"]
            r2a, r2as = preds_a["metrics"]["r2"]

            rnn_holdout_results.append({
                "Model": rnn_type.upper(),
                "Target": target_var,
                "Horizon": horizon,
                "RMSE Baseline": fmt(rb, rs),
                "RMSE Actor": fmt(ra, ras),
                "RMSE Δ": f"{rb - ra:.3f}",
                "MAE Baseline": fmt(mb, ms),
                "MAE Actor": fmt(ma, mas),
                "MAE Δ": f"{mb - ma:.3f}",
                "R² Baseline": fmt(r2b, r2bs),
                "R² Actor": fmt(r2a, r2as),
                "R² Δ": f"{r2a - r2b:.3f}",
            })

            rnn_holdout_predictions[(rnn_type.upper(), target_var, horizon, "baseline")] = preds_b
            rnn_holdout_predictions[(rnn_type.upper(), target_var, horizon, "actor")] = preds_a

rnn_holdout_df = pd.DataFrame(rnn_holdout_results).sort_values(["Model", "Target", "Horizon"]).reset_index(drop=True)
print(rnn_holdout_df.to_markdown(index=False))

tuning

In [ ]:
import numpy as np
import pandas as pd
from itertools import product
import tensorflow as tf
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Bidirectional, GRU, LSTM, Dropout, MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D, Dense)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import ttest_rel, sem, t

# Assume this function exists and works correctly
#from feature_engineering import feature_engineering

import os
import random

def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()

set_seed(42)



def build_attention_model(input_shape, rnn_units=128, dense_units=64, dropout=0.3, rnn_type='gru',
                          conv_filters=64, kernel_size=3, pool_size=2):
    input_layer = Input(shape=input_shape)
    x = Conv1D(filters=conv_filters, kernel_size=kernel_size, activation='relu', padding='same')(input_layer)
    x = MaxPooling1D(pool_size=pool_size)(x)
    x = Bidirectional(GRU(rnn_units, return_sequences=True) if rnn_type == 'gru' else LSTM(rnn_units, return_sequences=True))(x)
    x = Dropout(dropout)(x)
    x = (GRU(dense_units, return_sequences=True) if rnn_type == 'gru' else LSTM(dense_units, return_sequences=True))(x)
    x = Dropout(dropout)(x)
    x = MultiHeadAttention(num_heads=4, key_dim=32)(x, x)
    x = LayerNormalization(epsilon=1e-6)(x)
    x = GlobalAveragePooling1D()(x)
    output = Dense(1)(x)
    model = Model(inputs=input_layer, outputs=output)
    model.compile(optimizer='adam', loss='mse')
    return model


def create_seq_data(df, features, time_steps=15, target_col='target'):
    X, y = [], []
    for i in range(time_steps, len(df) - 1):
        X.append(df[features].iloc[i - time_steps:i].values)
        y.append(df[target_col].iloc[i + 1])
    return np.array(X), np.array(y)


def train_attention_rnn(df, features, rnn_type='gru', time_steps=15, n_splits=5, custom_params=None):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    rmse_list, mae_list = [], []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
        print(f"\n🔁 Fold {fold + 1}")

        df_train_raw = df.iloc[:test_idx[0]].copy()
        df_test_raw = df.iloc[test_idx].copy()

        df_train_fe, _, _ = feature_engineering(df_train_raw, residual_target=True)
        df_context = pd.concat([df_train_raw.tail(40), df_test_raw])
        df_test_fe, _, _ = feature_engineering(df_context, residual_target=True)
        df_test_fe = df_test_fe.loc[df_test_raw.index.intersection(df_test_fe.index)]

        df_train_fe.dropna(inplace=True)
        df_test_fe.dropna(inplace=True)

        scaler_x = StandardScaler()
        scaler_y = StandardScaler()
        df_train_fe[features] = scaler_x.fit_transform(df_train_fe[features])
        df_test_fe[features] = scaler_x.transform(df_test_fe[features])
        df_train_fe['target_scaled'] = scaler_y.fit_transform(df_train_fe[['target']])
        df_test_fe['target_scaled'] = scaler_y.transform(df_test_fe[['target']])

        X_train, y_train = create_seq_data(df_train_fe, features, time_steps, 'target_scaled')
        X_test, y_test = create_seq_data(df_test_fe, features, time_steps, 'target_scaled')

        base_values = df_test_fe['TT'].iloc[time_steps + 1: time_steps + 1 + len(y_test)].reset_index(drop=True)

        model = build_attention_model(
            input_shape=(X_train.shape[1], X_train.shape[2]),
            rnn_units=custom_params.get('rnn_units', 128),
            dense_units=custom_params.get('dense_units', 64),
            dropout=custom_params.get('dropout', 0.3),
            rnn_type=rnn_type,
            conv_filters=custom_params.get('conv_filters', 64),
            kernel_size=custom_params.get('kernel_size', 3),
            pool_size=custom_params.get('pool_size', 2)
         )

        callbacks = [
            EarlyStopping(patience=5, restore_best_weights=True),
            ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-5)
        ]

        model.fit(X_train, y_train, validation_data=(X_test, y_test),
                  epochs=100, batch_size=custom_params.get('batch_size', 16),
                  callbacks=callbacks, verbose=0)

        y_pred_scaled = model.predict(X_test).flatten()
        y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
        y_test_inv = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

        y_pred_final = base_values + y_pred
        y_true_final = base_values + y_test_inv

        rmse = np.sqrt(mean_squared_error(y_true_final, y_pred_final))
        mae = mean_absolute_error(y_true_final, y_pred_final)
        rmse_list.append(rmse)
        mae_list.append(mae)

    return rmse_list, mae_list


# Load data
print("\n📦 Loading data and setting up parameter grid")
df = df_train_full.copy()
df_model, baseline_features, actor_features = feature_engineering(df, residual_target=True)

param_grid = {
    'rnn_units': [64, 128],
    'dense_units': [32, 64],
    'dropout': [0.2, 0.3],
    'batch_size': [16, 32],
    'conv_filters': [32, 64],
    'kernel_size': [3, 5],
    'pool_size': [2, 3]
}
param_combos = list(product(*param_grid.values()))
param_names = list(param_grid.keys())

results_summary = []

for i, combo in enumerate(param_combos):
    set_seed(42)
    params = dict(zip(param_names, combo))
    print(f"\n\n🔧 Config {i+1}/{len(param_combos)}: {params}")

    rmse_b, mae_b = train_attention_rnn(df, baseline_features, rnn_type='gru', custom_params=params)
    rmse_a, mae_a = train_attention_rnn(df, baseline_features + actor_features, rnn_type='gru', custom_params=params)

    delta = np.mean(rmse_b) - np.mean(rmse_a)
    results_summary.append({
        'params': params,
        'rmse_baseline': np.mean(rmse_b),
        'rmse_actor': np.mean(rmse_a),
        'delta': delta,
        'outperforms': delta > 0
    })

    print(f"✅ Δ RMSE = {delta:.4f} | {'Actor Wins ✅' if delta > 0 else 'Baseline Wins ❌'}")

# Summary
summary_df = pd.DataFrame(results_summary).sort_values(by='delta', ascending=False)
print("\n\n📋 Final Results Summary:")
print(summary_df[['params', 'rmse_baseline', 'rmse_actor', 'delta', 'outperforms']].to_markdown(index=False))

print("\n✅ Configurations where Actor-Enriched outperformed Baseline:")
print(summary_df[summary_df['outperforms']][['params', 'rmse_baseline', 'rmse_actor', 'delta']].to_markdown(index=False))


In [ ]:
print("\n✅ Configurations where Actor-Enriched outperformed Baseline:")
print(summary_df[summary_df['outperforms']][['params', 'rmse_baseline', 'rmse_actor', 'delta']].to_markdown(index=False))
#choose mostly appearing configurations -> rnn_units 64 , dense 32, dropout 0.2 batch size 16

In [ ]:
from itertools import product
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, Bidirectional,
    GRU, LSTM, Dropout, MultiHeadAttention, LayerNormalization,
    GlobalAveragePooling1D, Dense
)

import os
import random

def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()

set_seed(42)

def build_attention_model(input_shape, rnn_units=128, dense_units=64, dropout=0.3, rnn_type='gru',
                          conv_filters=64, kernel_size=3, pool_size=2):
    input_layer = Input(shape=input_shape)
    x = Conv1D(filters=conv_filters, kernel_size=kernel_size, activation='relu', padding='same')(input_layer)
    x = MaxPooling1D(pool_size=pool_size)(x)
    x = Bidirectional(GRU(rnn_units, return_sequences=True) if rnn_type == 'gru' else LSTM(rnn_units, return_sequences=True))(x)
    x = Dropout(dropout)(x)
    x = (GRU(dense_units, return_sequences=True) if rnn_type == 'gru' else LSTM(dense_units, return_sequences=True))(x)
    x = Dropout(dropout)(x)
    x = MultiHeadAttention(num_heads=4, key_dim=32)(x, x)
    x = LayerNormalization(epsilon=1e-6)(x)
    x = GlobalAveragePooling1D()(x)
    output = Dense(1)(x)
    model = Model(inputs=input_layer, outputs=output)
    model.compile(optimizer='adam', loss='mse')
    return model

# === Sequence Creator ===
def create_seq_data(df, features, time_steps=15, target_col='target'):
    X, y = [], []
    for i in range(time_steps, len(df) - 1):
        X.append(df[features].iloc[i - time_steps:i].values)
        y.append(df[target_col].iloc[i + 1])
    return np.array(X), np.array(y)

# === Training Function ===
def train_attention_rnn(df, features, rnn_type='lstm', time_steps=15, n_splits=5, custom_params=None):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    rmse_list, mae_list = [], []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
        print(f"\n🔁 Fold {fold + 1}")

        df_train_raw = df.iloc[:test_idx[0]].copy()
        df_test_raw = df.iloc[test_idx].copy()

        df_train_fe, _, _ = feature_engineering(df_train_raw, residual_target=True)
        df_context = pd.concat([df_train_raw.tail(40), df_test_raw])
        df_test_fe, _, _ = feature_engineering(df_context, residual_target=True)
        df_test_fe = df_test_fe.loc[df_test_raw.index.intersection(df_test_fe.index)]

        df_train_fe.dropna(inplace=True)
        df_test_fe.dropna(inplace=True)

        scaler_x = StandardScaler()
        scaler_y = StandardScaler()
        df_train_fe[features] = scaler_x.fit_transform(df_train_fe[features])
        df_test_fe[features] = scaler_x.transform(df_test_fe[features])
        df_train_fe['target_scaled'] = scaler_y.fit_transform(df_train_fe[['target']])
        df_test_fe['target_scaled'] = scaler_y.transform(df_test_fe[['target']])

        X_train, y_train = create_seq_data(df_train_fe, features, time_steps, 'target_scaled')
        X_test, y_test = create_seq_data(df_test_fe, features, time_steps, 'target_scaled')
        base_values = df_test_fe['TT'].iloc[time_steps + 1: time_steps + 1 + len(y_test)].reset_index(drop=True)

        model = build_attention_model(
            input_shape=(X_train.shape[1], X_train.shape[2]),
            rnn_units=custom_params.get('rnn_units', 128),
            dense_units=custom_params.get('dense_units', 64),
            dropout=custom_params.get('dropout', 0.3),
            rnn_type=rnn_type,
            conv_filters=custom_params.get('conv_filters', 64),
            kernel_size=custom_params.get('kernel_size', 3),
            pool_size=custom_params.get('pool_size', 2)
         )

        callbacks = [
            EarlyStopping(patience=5, restore_best_weights=True),
            ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-5)
        ]

        model.fit(X_train, y_train, validation_data=(X_test, y_test),
                  epochs=100, batch_size=custom_params.get('batch_size', 16),
                  callbacks=callbacks, verbose=0)

        y_pred_scaled = model.predict(X_test).flatten()
        y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
        y_test_inv = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

        y_pred_final = base_values + y_pred
        y_true_final = base_values + y_test_inv

        rmse = np.sqrt(mean_squared_error(y_true_final, y_pred_final))
        mae = mean_absolute_error(y_true_final, y_pred_final)
        rmse_list.append(rmse)
        mae_list.append(mae)

    return rmse_list, mae_list

# === Run LSTM Grid Search ===
print("\n📦 Running LSTM hyperparameter tuning")

df = df_train_full.copy()
df_model, baseline_features, actor_features = feature_engineering(df, residual_target=True)

param_grid = {
    'rnn_units': [64, 128],
    'dense_units': [32, 64],
    'dropout': [0.2, 0.3],
    'batch_size': [16, 32],
    'conv_filters': [32, 64],
    'kernel_size': [3, 5],
    'pool_size': [2, 3]
}
param_combos = list(product(*param_grid.values()))
param_names = list(param_grid.keys())

results_summary = []

for i, combo in enumerate(param_combos):
    set_seed(42)
    params = dict(zip(param_names, combo))
    print(f"\n\n🔧 Config {i+1}/{len(param_combos)} | Params: {params}")

    rmse_b, mae_b = train_attention_rnn(df, baseline_features, rnn_type='lstm', custom_params=params)
    rmse_a, mae_a = train_attention_rnn(df, baseline_features + actor_features, rnn_type='lstm', custom_params=params)

    delta = np.mean(rmse_b) - np.mean(rmse_a)
    results_summary.append({
        'params': params,
        'rmse_baseline': np.mean(rmse_b),
        'rmse_actor': np.mean(rmse_a),
        'delta': delta,
        'outperforms': delta > 0
    })

    print(f"✅ Δ RMSE = {delta:.4f} | {'Actor Wins ✅' if delta > 0 else 'Baseline Wins ❌'}")

# === Print Summary ===
summary_df = pd.DataFrame(results_summary).sort_values(by='delta', ascending=False)
print("\n\n📋 LSTM Tuning Summary:")
print(summary_df[['params', 'rmse_baseline', 'rmse_actor', 'delta', 'outperforms']].to_markdown(index=False))

print("\n✅ Actor-Enriched configurations that outperformed Baseline:")
print(summary_df[summary_df['outperforms']][['params', 'rmse_baseline', 'rmse_actor', 'delta']].to_markdown(index=False))


In [ ]:
print("\n✅ Actor-Enriched configurations that outperformed Baseline:")
print(summary_df[summary_df['outperforms']][['params', 'rmse_baseline', 'rmse_actor', 'delta']].to_markdown(index=False))
#choose mostly appearing configurations -> rnn_units 64 , dense 32, dropout 0.2 batch size 16

# Final Plot


In [ ]:
pred_b_lgbm = final_model_b.predict(X_test_b) + base_values
pred_a_lgbm = final_model_a.predict(X_test_a) + base_values


In [ ]:
import matplotlib.pyplot as plt

# Colorblind-safe distinct colors
colors = {
    'XGBoost': '#E69F00',        # Mustard
    'LightGBM': '#56B4E9',       # Light Blue
    'LSTM No Attn': '#009E73',   # Green
    'LSTM Attn': '#332288',      # Dark Blue
    'GRU No Attn': '#CC79A7',    # Light Purple
    'GRU Attn': '#999999'        # Gray
}

# Setup figure
plt.figure(figsize=(18, 10))

# Plot actual values
plt.plot(df_test_fe.index[1:], y_true, label='Actual', color='black', linewidth=3)

# Actor-enriched predictions
plt.plot(df_test_fe.index[1:][:len(pred_a)], pred_a, '--', label='XGBoost Actor-Enriched', color=colors['XGBoost'], linewidth=2.5)
plt.plot(df_test_fe.index[1:][:len(pred_a_lgbm)], pred_a_lgbm, '--', label='LightGBM Actor-Enriched', color=colors['LightGBM'], linewidth=2.5)

plt.plot(preds_final_lstm_a_noatt['index'], preds_final_lstm_a_noatt['y_pred'], ':', label='LSTM Actor-Enriched (No Attn)', color=colors['LSTM No Attn'], linewidth=2.5, marker='o', markevery=10)
plt.plot(preds_final_lstm_a['index'], preds_final_lstm_a['y_pred'], '-', label='LSTM Actor-Enriched (Attn)', color=colors['LSTM Attn'], linewidth=2.5)

plt.plot(preds_final_gru_a_noatt['index'], preds_final_gru_a_noatt['y_pred'], ':', label='GRU Actor-Enriched (No Attn)', color=colors['GRU No Attn'], linewidth=2.5, marker='s', markevery=10)
plt.plot(preds_final_gru_a['index'], preds_final_gru_a['y_pred'], '-', label='GRU Actor-Enriched (Attn)', color=colors['GRU Attn'], linewidth=2.5)

# Final styling with increased font sizes
plt.title("Final Holdout Test: Actor-Enriched Model Predictions vs Actual", fontsize=40)
plt.xlabel("Time", fontsize=40)
plt.ylabel("TT", fontsize=40)
plt.xticks(fontsize=40)
plt.yticks(fontsize=40)
plt.legend(loc='upper left', ncol=2, fontsize=25, frameon=True)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

import matplotlib.pyplot as plt

plt.figure(figsize=(18, 10))

# Actual
plt.plot(df_test_fe.index[1:], y_true, label='Actual', color='black', linewidth=3)

# Actor-enriched model predictions
plt.plot(df_test_fe.index[1:][:len(pred_a)], pred_a, '--', label='XGBoost Actor-Enriched', color=colors['XGBoost'], linewidth=2.5)
plt.plot(df_test_fe.index[1:][:len(pred_a_lgbm)], pred_a_lgbm, '--', label='LightGBM Actor-Enriched', color=colors['LightGBM'], linewidth=2.5)
plt.plot(preds_final_lstm_a_noatt['index'], preds_final_lstm_a_noatt['y_pred'], ':', label='LSTM Actor-Enriched (No Attn)', color=colors['LSTM No Attn'], linewidth=2.5, marker='o', markevery=10)
plt.plot(preds_final_lstm_a['index'], preds_final_lstm_a['y_pred'], '-', label='LSTM Actor-Enriched (Attn)', color=colors['LSTM Attn'], linewidth=2.5)
plt.plot(preds_final_gru_a_noatt['index'], preds_final_gru_a_noatt['y_pred'], ':', label='GRU Actor-Enriched (No Attn)', color=colors['GRU No Attn'], linewidth=2.5, marker='s', markevery=10)
plt.plot(preds_final_gru_a['index'], preds_final_gru_a['y_pred'], '-', label='GRU Actor-Enriched (Attn)', color=colors['GRU Attn'], linewidth=2.5)

# Styling
plt.title("Final Holdout Test: Actor-Enriched Model Predictions vs Actual", fontsize=40)
plt.xlabel("Time", fontsize=40)
plt.ylabel("TT", fontsize=40)
plt.xticks(fontsize=40)
plt.yticks(fontsize=40)
plt.legend(loc='upper left', ncol=2, fontsize=25, frameon=True)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

# ✅ Save the figure *before* showing it
plt.savefig("test_set_results_bpic2017.pdf", format='pdf')

# Show it
plt.show()

# ✅ Download it
from google.colab import files
files.download("test_set_results_bpic2017.pdf")




In [ ]:
from google.colab import files
files.download("test set results bpic2017.pdf")

# ARIMA

In [ ]:
# --- Step 1: Feature Engineering (residual target)
df_train_fe, _, _ = feature_engineering(df_train_full, residual_target=True)

df_context = pd.concat([df_train_full.tail(40), df_test_final])
df_test_fe, _, _ = feature_engineering(df_context, residual_target=True)
df_test_fe = df_test_fe.loc[df_test_final.index.intersection(df_test_fe.index)]

df_train_fe.dropna(inplace=True)
df_test_fe.dropna(inplace=True)

# --- Step 2: Extract aligned residual target + base
base_values = df_test_fe['TT'].shift(1)
y_test_arima = df_test_fe['target']

df_eval = pd.DataFrame({
    'base': base_values,
    'target': y_test_arima
}).dropna().reset_index()

# Final aligned series
base_values = df_eval['base'].reset_index(drop=True)
y_test_arima = df_eval['target'].reset_index(drop=True)

# --- Step 3: Fit ARIMA on residuals
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error

print("📦 Training ARIMA(1,0,0) on residuals...")
y_train_arima = df_train_fe['target']
model_arima = ARIMA(y_train_arima, order=(1, 0, 0))
model_arima_fit = model_arima.fit()

# Forecast
resid_forecast = model_arima_fit.forecast(steps=len(y_test_arima))

# --- Step 4: Safe Alignment — enforce equal lengths
min_len = min(len(resid_forecast), len(base_values), len(y_test_arima))

y_pred = resid_forecast[:min_len].reset_index(drop=True) + base_values[:min_len]
y_true = base_values[:min_len] + y_test_arima[:min_len]

# --- Step 5: Evaluation
rmse_arima = np.sqrt(mean_squared_error(y_true, y_pred))
mae_arima = mean_absolute_error(y_true, y_pred)

print(f"\n📊 ARIMA Benchmark on Residuals → RMSE: {rmse_arima:.4f}, MAE: {mae_arima:.4f}")



In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df_eval['index'][:min_len], y_true, label='Actual', color='black')
plt.plot(df_eval['index'][:min_len], y_pred, '--', label='ARIMA Residual Forecast', color='purple')
plt.title("🔍 ARIMA Benchmark: Actual vs Reconstructed TT")
plt.xlabel("Time")
plt.ylabel("Duration")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
